Plate Reconstruction, Feature Extraction, and Conversion to World Builder JSON
------------------------------------------------------------------------------
This script reconstructs tectonic plate configurations at a chosen geological time
using GPlately and plate reconstruction models, either downloaded from a remote
repository or loaded from local files. It extracts geological features such as
continents, ridges, trenches, and miscellaneous boundaries, generates additional
derived datasets such as seafloor age, spreading rate, trench kinematics, slab
geometry proxies, and continental thickness information, and converts all of these
into a World Builder-compatible JSON file for geodynamic model initialization.

The workflow combines plate reconstruction, geospatial processing, raster generation,
and feature parameterization in order to build a physically informed starting model
that can be used directly in Geodynamic World Builder.

Workflow:
1. Load a plate reconstruction model from a remote plate-model repository or from
   user-provided local reconstruction files.
2. Reconstruct geological features at the selected geological age and save them as
   GeoJSON files.
3. Generate seafloor age and spreading rate grids from the reconstructed plate model.
4. Analyse trench geometries to estimate quantities such as average azimuth, slab dip,
   slab thickness, dip direction, ridge distance, spreading velocity, and subducting
   velocity.
5. Process continental geometries to derive merged continental polygons and approximate
   lithospheric thickness / passive margin structure.
6. Convert all extracted and derived features into a World Builder-compatible JSON file,
   including mantle layers, oceanic lithosphere, continental plates, and subducting slabs.
7. Save the final JSON output for direct use in GWB-based geodynamic simulations.

Usage Instructions:
- Set `reconstruction_time` to define the reconstruction age in Ma.
- Set `plate_reconstruction_model` to choose the reconstruction model to use.
- Set `use_local_files = True` to use local reconstruction files instead of downloading
  model data.
- Adjust the feature-inclusion flags to control which components are included in the final model.
- A companion installation script, `Installation_dependencies_gplatly2gwb.sh`, is
  provided to install dependencies with compatible versions.
- Required Python packages include `pygplates`, `gplately`, `matplotlib`, `cartopy`,
  `geopandas`, `numpy`, `scipy`, and `netCDF4`.

Notes:
- PyGMT is currently not working reliably in this workflow, even when installed with
  Python 3.11.
- This workflow relies on several simplifying assumptions and empirical relationships
  (e.g., slab dip, trench age, lithospheric thickness, velocity corrections). While
  these choices enable a fully automated pipeline, they introduce approximations that
  may not be physically consistent in all cases.
- Additional validation, refinement, and future improvements will be required to ensure
  robustness, physical consistency, and broader applicability of the generated models.
- Please refer to issue #833 (https://github.com/GeodynamicWorldBuilder/WorldBuilder/issues/833) on GitHub for ongoing development and discussion.

Outputs:
- GeoJSON files for reconstructed geological features.
- NetCDF grids for seafloor age and spreading rate.
- Enriched trench and continent GeoJSON files with derived parameters.
- A final World Builder JSON file ready for use in GWB.

In [ ]:
import sys
import os
import glob
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import json
import pygplates
import gplately
from plate_model_manager import PlateModelManager


In [ ]:
# User-defined parameters
reconstruction_time = 400  # Ma
plate_reconstruction_model = "Merdith2021" #Muller2019 #Clennett2020
use_local_files = False  # Set to True to use local files instead of online repo

#for use of local files use 
input_directory = "./path/to/local_plate_reconstruction/"


# Define the output directory
output_dir = "./data/"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Load plate model data based on user choice
if use_local_files:

    # Load rotation model
    rotation_filenames = glob.glob(os.path.join(input_directory, '*.rot'))
    rotation_model = pygplates.RotationModel(rotation_filenames)

    # Load topology features
    topology_filenames = glob.glob(os.path.join(input_directory, '*.gpml'))
    topology_features = pygplates.FeatureCollection()
    for topology_filename in topology_filenames:
        if "Inactive" not in topology_filename:
            topology_features.add(pygplates.FeatureCollection(topology_filename))

    # Load static polygons
    
    static_polygon_file = os.path.join(input_directory, "StaticGeometries/StaticPolygons/polygons.shp")
    static_polygons = pygplates.FeatureCollection(static_polygon_file)


    ## Example of use of local file 
    #  Additonnal information : https://gplates.github.io/gplately/v1.3.0/02-PlateReconstructions.html

    ## For testing download the following data set :
    ## https://www.earthbyte.org/webdav/ftp/Data_Collections/Muller_etal_2019_Tectonics/Muller_etal_2019_PlateMotionModel/

    ## For example case uncomment the following lines 
    # coastlines = input_directory+"StaticGeometries/Coastlines/Global_coastlines_2019_v1_low_res.shp"
    # continents = input_directory+"StaticGeometries/ContinentalPolygons/Global_EarthByte_GPlates_PresentDay_ContinentalPolygons_2019_v1.shp"
    # COBs = input_directory+"StaticGeometries/AgeGridInput/Global_EarthByte_GeeK07_IsoCOB_2019_v2.gpml"
    # input_directory = "./local_repository_where_downloaded/Muller_etal_2019_PlateMotionModel_v2.0_Tectonics/"
    # static_polygon_file = input_directory+"StaticGeometries/StaticPolygons/Global_EarthByte_GPlates_PresentDay_StaticPlatePolygons_2019_v1.shp"
        
else:
    # Load model data from remote repository
    pm_manager = PlateModelManager()
    model_data = pm_manager.get_model(plate_reconstruction_model, data_dir="plate-model-repo")
    rotation_model = model_data.get_rotation_model()
    topology_features = model_data.get_topologies()
    static_polygons = model_data.get_static_polygons()

    # Load coastline and continent data
    coastlines = model_data.get_layer('Coastlines')
    continents = model_data.get_layer('ContinentalPolygons')

# Create plate reconstruction model
model = gplately.PlateReconstruction(rotation_model, topology_features, static_polygons)

# Initialize PlotTopologies with the plate reconstruction model and data layers
gplot = gplately.PlotTopologies(model, coastlines=coastlines, continents=continents)

# Set reconstruction time
gplot.time = reconstruction_time  # Ma

# Define the default features to extract
default_features = {
    "continents": gplot.get_continents,
    "ridges_and_transforms": gplot.get_ridges_and_transforms,
    "trenches": gplot.get_trenches,
    "misc_boundaries": gplot.get_misc_boundaries,

}

# Extract and save the default features
for feature, func in default_features.items():
    try:
        geo_data = func()
        if geo_data is None or geo_data.empty:
            print(f"Feature '{feature}' is not available for this plate reconstruction model.")
        else:
            file_path = os.path.join(output_dir, f"{feature}.geojson")
            geo_data.to_file(file_path, driver="GeoJSON")
            print(f"Feature '{feature}' saved as {file_path}")
    except Exception as e:
        print(f"Error extracting '{feature}': {e}")


In [ ]:
# List all callable methods starting with 'get_'
get_methods = [method for method in dir(gplot) if callable(getattr(gplot, method)) and method.startswith('get_')]
print("Available 'get_' methods in PlotTopologies:")
for method in get_methods:
    print(method)

In [ ]:
#Plot all the features from gplately possible
# re-import dependencies if necessary
# import matplotlib.pyplot as plt
# import cartopy.crs as ccrs
# import os
# --- Generalized plotter for all get_/plot_ pairs ---
def plot_any_layer(feature_name, color="blue"):
    get_name = f"get_{feature_name}"
    plot_name = f"plot_{feature_name}"
    title = feature_name.replace("_", " ").title()
    filename = f"{plate_reconstruction_model}_{reconstruction_time}Ma_{feature_name}.png"

    try:
        # Get function references if available
        get_func = getattr(gplot, get_name, None)
        plot_func = getattr(gplot, plot_name, None)

        if get_func is None:
            print(f"❌ {get_name} not implemented.")
            return

        # Check if the feature returns data
        gdf = get_func()
        if gdf is None or gdf.empty:
            print(f"⚠️  No data for {feature_name} at {reconstruction_time} Ma.")
            return

        # If there's no plot function, skip plotting
        if plot_func is None:
            print(f"⚠️  {plot_name} does not exist. Skipping plot.")
            return

        # Create figure
        fig = plt.figure(figsize=(12, 6))
        ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())
        ax.set_global()
        # ax.coastlines()
        gplot.plot_continents(ax, facecolor='0.8')

        # Plot the feature
        plot_func(ax=ax, color=color, linewidth=1.5)
        ax.set_title(f"{title} - {plate_reconstruction_model} @ {reconstruction_time} Ma", fontsize=14)

        outpath = os.path.join(output_dir, filename)
        # plt.savefig(outpath, dpi=300, bbox_inches="tight")  # Uncomment to save
        print(f"✅ Plotted {title} → {outpath}")
        plt.show()

    except Exception as e:
        print(f"❌ Error plotting {feature_name}: {e}")

# --- Feature list to plot from gplately ---
features_to_plot = [
    "all_topological_sections",
    "all_topologies",
    "coastlines",
    "continent_ocean_boundaries",
    "continental_crusts",
    "continental_rifts",
    "continents",
    "extended_continental_crusts",
    "faults",
    "fracture_zones",
    "inferred_paleo_boundaries",
    "misc_boundaries",
    "misc_transforms",
    "orogenic_belts",
    "passive_continental_boundaries",
    "ridges",
    "ridges_and_transforms",
    "slab_edges",
    "subduction_direction",
    "sutures",
    "terrane_boundaries",
    "transforms",
    "transitional_crusts",
    "trenches",
    "unclassified_features",
]

# --- Set time ---
gplot.time = reconstruction_time

# --- Plot all features ---
for feat in features_to_plot:
    plot_any_layer(feat, color="blue")


### Seafloor age and spreading-rate grid generation

This section computes oceanic seafloor age and spreading-rate grids from the selected
plate reconstruction model over a defined time window ending at the chosen reconstruction
age. It first checks the valid temporal range of the reconstruction model, adjusts the
requested time window if needed, and updates the active reconstruction time in both the
plate model and plotting object.

A `SeafloorGrid` object is then created using GPlately to reconstruct oceanic crust
through time from topological plate boundaries and mid-ocean ridge seed points. The
resulting seafloor age and spreading-rate fields are exported as NetCDF files, checked
for existence, and loaded back as raster objects for later plotting, interpolation, and
feature parameterization in the World Builder conversion workflow.

In [ ]:
import sys
import os
import numpy as np
import pygplates
from netCDF4 import Dataset

# Optional: Slab-Dip
import slabdip
from slabdip import SlabDipper

# ============================================================
# User parameters
# ============================================================
#reconstruction_time = 700   # Ma (example)
spread_window = 10          # time window to compute spreading rate (older -> younger)
#plate_reconstruction_model = "Merdith2021"

output_dir = os.path.abspath("data")
os.makedirs(output_dir, exist_ok=True)

# ============================================================
# Load valid reconstruction times from DataServer
# ============================================================
gDownload = gplately.DataServer(plate_reconstruction_model)
t0, t1 = gDownload.get_valid_times()

# GPlately returns times in descending order sometimes → fix
model_min_time = min(t0, t1)   # youngest, usually 0 Ma
model_max_time = max(t0, t1)   # oldest, e.g. 1000 Ma

print(f"Available reconstruction times for {plate_reconstruction_model}: "
      f"{model_min_time} Ma to {model_max_time} Ma")

# ============================================================
# Define timestep window (OLDER = larger values)
# ============================================================
# e.g., for reconstruction_time = 700:
# window = 10 → run from 710 Ma → 700 Ma

max_time = reconstruction_time + spread_window   # older
min_time = reconstruction_time                   # younger

# Clamp to model limits
if max_time > model_max_time:
    print(f"⚠️  Requested max_time {max_time} > model_max {model_max_time}, clamping...")
    max_time = model_max_time

if min_time < model_min_time:
    print(f"⚠️  Requested min_time {min_time} < model_min {model_min_time}, clamping...")
    min_time = model_min_time

print(f"Final reconstruction window: {max_time} Ma → {min_time} Ma")

# ============================================================
# FIX: Must update the reconstruction model BEFORE SeafloorGrid
# ============================================================
model.time = reconstruction_time
gplot.time = reconstruction_time

# ============================================================
# Seafloor Grid parameters (same as original)
# ============================================================
ridge_time_step = 1.0
ridge_sampling = 0.5
grid_spacing = 0.25
refinement_levels = 6
initial_ocean_mean_spreading_rate = 50.0
extent = [-180, 180, -90, 90]

# ============================================================
# Create SeafloorGrid object
# ============================================================
seafloorgrid = gplately.oceans.SeafloorGrid(
    PlateReconstruction_object=model,
    PlotTopologies_object=gplot,
    max_time=max_time,
    min_time=min_time,
    ridge_time_step=ridge_time_step,
    ridge_sampling=ridge_sampling,
    grid_spacing=grid_spacing,
    extent=extent,
    save_directory=output_dir,
    file_collection=plate_reconstruction_model,
    refinement_levels=refinement_levels,
    initial_ocean_mean_spreading_rate=initial_ocean_mean_spreading_rate,
    resume_from_checkpoints=False,
    zval_names=['SPREADING_RATE']
)

# ============================================================
# Run reconstruction
# ============================================================
print(f"Running reconstruction from {max_time:.2f} → {min_time:.2f} Ma ...")
seafloorgrid.reconstruct_by_topologies()

# NetCDF outputs
seafloorgrid.lat_lon_z_to_netCDF("SEAFLOOR_AGE", unmasked=True)
seafloorgrid.lat_lon_z_to_netCDF("SPREADING_RATE", unmasked=True)

# ============================================================
# File lookup helper
# ============================================================
def choose_existing(path1, path2):
    return path1 if os.path.exists(path1) else (path2 if os.path.exists(path2) else None)

# Construct expected filenames
age_masked = f"{output_dir}/SEAFLOOR_AGE/{plate_reconstruction_model}_SEAFLOOR_AGE_grid_{reconstruction_time:.2f}Ma.nc"
age_unmasked = f"{output_dir}/SEAFLOOR_AGE/{plate_reconstruction_model}_SEAFLOOR_AGE_grid_unmasked_{reconstruction_time:.2f}Ma.nc"

spread_masked = f"{output_dir}/SPREADING_RATE/{plate_reconstruction_model}_SPREADING_RATE_grid_{reconstruction_time:.2f}Ma.nc"
spread_unmasked = f"{output_dir}/SPREADING_RATE/{plate_reconstruction_model}_SPREADING_RATE_grid_unmasked_{reconstruction_time:.2f}Ma.nc"

age_file = choose_existing(age_masked, age_unmasked)
spread_file = choose_existing(spread_masked, spread_unmasked)

if age_file is None or spread_file is None:
    print("❌ Files not found. Contents of data directory:")
    for root, dirs, files in os.walk(output_dir):
        for f in files:
            print(" →", os.path.join(root, f))
    raise FileNotFoundError("Missing output grids!")

print(f"Seafloor AGE grid:      {age_file}")
print(f"Seafloor SPREAD grid:   {spread_file}")

# Inspect NetCDF structure
with Dataset(spread_file, "r") as ncfile:
    print("NetCDF Variables:", ncfile.variables.keys())

# Load grids as Raster objects
age_grid = gplately.grids.Raster(age_file)
spread_grid = gplately.grids.Raster(spread_file)

print("✅ Loaded both grids successfully.")


In [ ]:
# Plotting Seafloor Age grid
fig_age = plt.figure(figsize=(16, 12))
ax_age = fig_age.add_subplot(111, projection=ccrs.Mollweide(central_longitude=20))

# Plot the Seafloor Age grid
im1 = gplot.plot_grid(ax_age, age_grid.data, cmap='YlGnBu', vmin=0, vmax=200)
fig_age.colorbar(im1, orientation='horizontal', shrink=0.4, pad=0.1, label=f'Seafloor Age (Ma)')

# Set the plot title for Seafloor Age
plt.title(f"Seafloor Age for {plate_reconstruction_model} at {reconstruction_time} Ma")
plt.savefig(os.path.join(output_dir, f"{plate_reconstruction_model}_{reconstruction_time}Ma_seafloor_age.png"), dpi=300)
plt.show()

# Plotting Spreading Rate grid
fig_sr = plt.figure(figsize=(16, 12))
ax_sr = fig_sr.add_subplot(111, projection=ccrs.Mollweide(central_longitude=20))

# Plot the Spreading Rate grid
im2 = gplot.plot_grid(ax_sr, spread_grid.data, cmap='magma', alpha=0.7)
fig_sr.colorbar(im2, orientation='horizontal', shrink=0.4, pad=0.1, label=f'Spreading Rate (mm/yr)')

# Set the plot title for Spreading Rate
plt.title(f"Spreading Rate for {plate_reconstruction_model} at {reconstruction_time} Ma")
plt.savefig(os.path.join(output_dir, f"{plate_reconstruction_model}_{reconstruction_time}Ma_spreading_rate.png"), dpi=300)
plt.show()



# Slab Dip Calculation

**Slab dip**  is estimated using numerical models that incorporate ocean age grids and spreading rate data.

Installation instructions are available at [**Slab-Dip**](https://github.com/brmather/Slab-Dip).

For installation use pip install git+https://github.com/brmather/Slab-Dip.git@c2f2fe1a4378d25cd76c918b371047916602122d

Reference: [slabdip/predictor.py](https://github.com/brmather/Slab-Dip/blob/main/slabdip/predictor.py)

Sea floor age calculation example: 
https://github.com/GPlates/gplately/blob/master/Notebooks/10-SeafloorGrids.ipynb

### Trench Processing and Parameter Extraction

The following script processes trench geometries from a GeoJSON file and computes key geodynamic parameters using plate reconstruction data.

### Workflow

- Load trench geometries (`trenches.geojson`)
- Compute trench azimuth from segment orientations
- Sample subduction velocities using GPlately (`tessellate_subduction_zones`)
- Estimate slab dip and thickness via nearest-neighbor queries (cKDTree)
- Compute trench-averaged properties:
  - azimuth (°)
  - slab dip (°)
  - slab thickness (km, min 30 km)
- Store results in GeoJSON properties

This script enriches the trench GeoJSON file with additional parameters needed for slab and thermal model generation. For each trench segment, it computes a dipping point located 1000 km along the trench-normal direction, identifies the nearest ridge coordinates, estimates ridge-to-trench distance, samples spreading rate from the reconstructed seafloor grid, and calculates average subducting plate velocity from GPlately plate motions. The resulting dipping points, ridge coordinates, spreading velocities, subducting velocities, and ridge-to-trench distances are written back into the trench GeoJSON for later use in World Builder slab parameterization.

In [ ]:
import json
import numpy as np
from scipy.spatial import cKDTree
import gplately

# ----------------------------------
# Load trench GeoJSON
# ----------------------------------
file_path = "./data/trenches.geojson"
print(f"Processing file: {file_path}")

with open(file_path, 'r') as f:
    trenches_data = json.load(f)

# ----------------------------------
# Helper: azimuth calculation
# ----------------------------------
def calculate_azimuth(lon1, lat1, lon2, lat2):
    dlon = np.radians(lon2 - lon1)
    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)

    x = np.sin(dlon) * np.cos(lat2)
    y = (np.cos(lat1)*np.sin(lat2)
         - np.sin(lat1)*np.cos(lat2)*np.cos(dlon))

    azimuth = np.degrees(np.arctan2(x, y))
    return (azimuth + 360) % 360

# ----------------------------------
# Placeholder slab dip dataset
# ----------------------------------
slab_dip_data = {
    "lon": [165.186, 163.307, 160.950, 158.961],
    "lat": [-28.940, -25.426, -21.880, -19.193],
    "slab_dip": [30, 35, 40, 45],
    "slab_thickness": [100000, 90000, 80000, 75000]
}

slab_xyz = np.vstack(
    gplately.tools.lonlat2xyz(
        np.array(slab_dip_data["lon"]),
        np.array(slab_dip_data["lat"]),
        degrees=True
    )
).T

slab_tree = cKDTree(slab_xyz)

# ----------------------------------
# Load subduction data from GPlately
# ----------------------------------
time = reconstruction_time
sub = model.tessellate_subduction_zones(time)

sub_lons = sub[:, 0]
sub_lats = sub[:, 1]
sub_vels = sub[:, 2]

sub_xyz = np.vstack(
    gplately.tools.lonlat2xyz(sub_lons, sub_lats, degrees=True)
).T

sub_tree = cKDTree(sub_xyz)

# ----------------------------------
# MAIN LOOP
# ----------------------------------
for feature in trenches_data["features"]:

    geom = feature["geometry"]
    gtype = geom["type"]
    coords_raw = geom["coordinates"]

    # Normalize geometry
    if gtype == "LineString":
        segments = [np.array(coords_raw)]
    elif gtype == "MultiLineString":
        segments = [np.array(seg) for seg in coords_raw]
    else:
        continue

    # Collect trench-wide arrays
    trench_lons_all = []
    trench_lats_all = []
    subduction_records_all = []   # list of [sub_lon, sub_lat, sub_vel]

    # Compute azimuth over ALL segments
    azimuth_list = []

    for seg in segments:
        lons = seg[:, 0]
        lats = seg[:, 1]

        # Collect lon/lat
        trench_lons_all.extend(lons.tolist())
        trench_lats_all.extend(lats.tolist())

        # ------ Compute azimuth ------
        for i in range(len(lons) - 1):
            az = calculate_azimuth(lons[i], lats[i], lons[i+1], lats[i+1])
            azimuth_list.append(az)

        # ------ Subduction data sampling ------
        trench_xyz = np.vstack(
            gplately.tools.lonlat2xyz(lons, lats, degrees=True)
        ).T

        _, idx = sub_tree.query(trench_xyz)

        seg_sub_list = np.column_stack([
            sub_lons[idx],
            sub_lats[idx],
            sub_vels[idx]
        ])

        for row in seg_sub_list:
            subduction_records_all.append([
                float(row[0]), float(row[1]), float(row[2])
            ])

    # Compute averages
    avg_azimuth = float(np.mean(azimuth_list))

    # --- slab dip sampling (use all points) ---
    trench_xyz_full = np.vstack(
        gplately.tools.lonlat2xyz(
            np.array(trench_lons_all),
            np.array(trench_lats_all),
            degrees=True
        )
    ).T

    _, dip_idx = slab_tree.query(trench_xyz_full)

    avg_slab_dip = float(np.mean(np.array(slab_dip_data["slab_dip"])[dip_idx]))
    avg_slab_thick = float(
        max(
            np.mean(np.array(slab_dip_data["slab_thickness"])[dip_idx]),
            30000
        ) / 1000
    )

    # ----------------------------------
    # Write into GeoJSON
    # ----------------------------------
    props = feature["properties"]
    props["Trench_Longitudes"] = trench_lons_all
    props["Trench_Latitudes"] = trench_lats_all
    props["Subduction_Data"] = subduction_records_all

    props["Average_Azimuth"] = round(avg_azimuth, 2)
    props["Average_Slab_Dip"] = round(avg_slab_dip, 2)
    props["Average_Slab_Thickness_km"] = round(avg_slab_thick, 2)

# ----------------------------------
# Save output
# ----------------------------------
out_file = "./data/trenches_with_avg_parameters.geojson"
with open(out_file, "w") as f:
    json.dump(trenches_data, f, indent=4)

print(f"✅ Updated GeoJSON saved to {out_file}")


This script adds additional slab and kinematic parameters to the trench GeoJSON file. For each trench segment, it computes a dipping point from the trench-normal azimuth, identifies nearby ridge coordinates, estimates ridge-to-trench distance, samples spreading rate from the reconstructed spreading-rate grid, and calculates subducting plate velocity from GPlately plate motions. These values are then saved back into the trench GeoJSON for later use in World Builder slab parameterization.


In [ ]:
import numpy as np
import json
import os
from scipy.spatial import cKDTree
import gplately


# -------------------------------------------
# CONSTANTS
# -------------------------------------------
MIN_VELOCITY = 0.001                  # m/yr
MIN_DISTANCE_KM = 50                  # km
DEFAULT_SPREADING_VELOCITY = 0.05     # m/yr
EARTH_DEG_TO_KM = 111                 # 1° ≈ 111 km


# -------------------------------------------
# JSON-SAFE CLEANER
# -------------------------------------------
def clean_value(v):
    """Convert to Python float and remove NaN/Inf."""
    try:
        v = float(v)
    except Exception:
        return None
    if np.isnan(v) or np.isinf(v):
        return None
    return v


# -------------------------------------------
# DIPPING POINT FUNCTION (RESTORED)
# -------------------------------------------
def calculate_dipping_point_with_polarity(mid_lon, mid_lat, trench_azimuth):
    """
    Compute dipping point 1000 km along the trench normal direction.
    """
    dip_distance_km = 1000.0

    az = np.radians(trench_azimuth)

    # km → degrees
    delta_lon = (dip_distance_km * np.sin(az)) / 111.32
    delta_lat = (dip_distance_km * np.cos(az)) / 110.54

    dip_lon = mid_lon + delta_lon
    dip_lat = mid_lat + delta_lat

    return [round(float(dip_lon), 5), round(float(dip_lat), 5)]


# -------------------------------------------
# LOAD TRENCH GEOJSON
# -------------------------------------------
file_path = "./data/trenches_with_avg_parameters.geojson"
print(f"Processing file: {file_path}")

with open(file_path, "r") as f:
    trenches_data = json.load(f)


# -------------------------------------------
# LOAD SUBDUCTION AZIMUTH FROM GPLATELY
# -------------------------------------------
subduction_data = model.tessellate_subduction_zones(reconstruction_time)
sub_lons = subduction_data[:, 0]
sub_lats = subduction_data[:, 1]
sub_azimuths = subduction_data[:, 7]   # trench normal azimuth


# -------------------------------------------
# LOAD RIDGE GEOMETRY
# -------------------------------------------
ridge_file_path = "./data/ridges_and_transforms.geojson"
print(f"Processing ridge file: {ridge_file_path}")

with open(ridge_file_path, "r") as f:
    ridge_data = json.load(f)

ridge_points = []
for feat in ridge_data["features"]:
    geom = feat["geometry"]
    gtype = geom["type"]
    if gtype == "LineString":
        ridge_points.extend(geom["coordinates"])
    elif gtype == "MultiLineString":
        for seg in geom["coordinates"]:
            ridge_points.extend(seg)

ridge_points = np.array(ridge_points)
ridge_tree = cKDTree(ridge_points)


# -------------------------------------------
# LOAD SPREADING RATE GRID (masked / unmasked)
# -------------------------------------------
spread_masked = (
    f"{output_dir}/SPREADING_RATE/"
    f"{plate_reconstruction_model}_SPREADING_RATE_grid_{reconstruction_time:.2f}Ma.nc"
)
spread_unmasked = (
    f"{output_dir}/SPREADING_RATE/"
    f"{plate_reconstruction_model}_SPREADING_RATE_grid_unmasked_{reconstruction_time:.2f}Ma.nc"
)

if os.path.exists(spread_masked):
    spread_file = spread_masked
elif os.path.exists(spread_unmasked):
    spread_file = spread_unmasked
else:
    raise FileNotFoundError(
        f"❌ Spreading rate grid not found in:\n  {spread_masked}\n  {spread_unmasked}"
    )

print(f"Using spreading rate grid: {spread_file}")
spreadrate_grid = gplately.grids.Raster(spread_file)


# -------------------------------------------
# MAIN LOOP
# -------------------------------------------
for feature in trenches_data["features"]:

    geom = feature["geometry"]
    gtype = geom["type"]
    coords = geom["coordinates"]

    if gtype == "LineString":
        segments = [np.array(coords)]
    elif gtype == "MultiLineString":
        segments = [np.array(s) for s in coords]
    else:
        # unsupported geometry
        continue

    # Lists for output
    ridge_list = []
    spread_list = []
    sub_list = []
    dist_list = []

    # -------------------------------------------
    # PROCESS EACH SEGMENT
    # -------------------------------------------
    for seg in segments:

        # ---- midpoint ----
        mid = len(seg) // 2
        mid_lon, mid_lat = seg[mid]

        # ---- nearest trench normal azimuth ----
        idx = np.argmin((sub_lons - mid_lon) ** 2 + (sub_lats - mid_lat) ** 2)
        trench_azimuth = sub_azimuths[idx]

        # ---- dipping point (stored once per feature, last segment wins for MultiLineString) ----
        dipping_point = calculate_dipping_point_with_polarity(mid_lon, mid_lat, trench_azimuth)
        feature.setdefault("properties", {})
        feature["properties"]["Dipping_Point"] = dipping_point

        # ---- nearest ridges ----
        _, ridge_idx = ridge_tree.query([mid_lon, mid_lat], k=5)
        closest = ridge_points[ridge_idx]

        # ---- ridge distance ----
        dist_deg = np.linalg.norm(closest - np.array([mid_lon, mid_lat]), axis=1)
        dist_km = float(np.min(dist_deg) * EARTH_DEG_TO_KM)

        # ---- spreading rate ----
        spread_vals = []
        for lon, lat in closest:
            v = spreadrate_grid.interpolate(float(lon), float(lat))
            if np.isnan(v):
                v = DEFAULT_SPREADING_VELOCITY
            spread_vals.append(float(v) * 0.01)  # cm/yr → m/yr

        avg_spread = float(np.mean(spread_vals))

        # Enforce minimum ridge–trench distance
        if dist_km < MIN_DISTANCE_KM and dist_km > 0:
            avg_spread *= (MIN_DISTANCE_KM / dist_km)

        if np.isnan(avg_spread) or np.isinf(avg_spread):
            avg_spread = DEFAULT_SPREADING_VELOCITY

        # ---- subducting velocity from plate velocities ----
        gpts = gplately.Points(model, seg[:, 0].tolist(), seg[:, 1].tolist())
        velx, vely = gpts.plate_velocity(reconstruction_time)
        speed = np.sqrt(velx**2 + vely**2)
        sub_vel = float(np.mean(speed))

        if sub_vel < MIN_VELOCITY or np.isnan(sub_vel) or np.isinf(sub_vel):
            sub_vel = MIN_VELOCITY

        # ---- append cleaned values ----
        ridge_list.append([[float(lon), float(lat)] for lon, lat in closest])
        spread_list.append(clean_value(round(avg_spread, 5)))
        sub_list.append(clean_value(round(sub_vel, 5)))
        dist_list.append(clean_value(round(dist_km, 3)))

        # Debug
        print(f"\nTrench {feature.get('id', 'N/A')}:")
        print(f"  Midpoint:       {mid_lon:.3f}, {mid_lat:.3f}")
        print(f"  Azimuth:        {float(trench_azimuth):.2f}°")
        print(f"  Dipping point:  {dipping_point}")
        print(f"  Closest ridges: {closest.tolist()}")
        print(f"  Spread vel:     {avg_spread} m/yr")
        print(f"  Subduction vel: {sub_vel} m/yr")
        print(f"  Min dist:       {dist_km} km")
        print("-" * 60)

    # -------------------------------------------
    # SAVE TO PROPERTIES
    # -------------------------------------------
    props = feature["properties"]
    props["Ridge_Coordinates"] = ridge_list
    props["Spreading_Velocity_m_per_yr"] = spread_list
    props["Subducting_Velocity_m_per_yr"] = sub_list
    props["Minimum_Ridge_to_Trench_Distance_km"] = dist_list


# -------------------------------------------
# WRITE FILE
# -------------------------------------------
with open(file_path, "w") as f:
    json.dump(trenches_data, f, indent=4)

print("\n✅ Finished: dipping points + spreading + subducting + ridge distance saved.")


The following section defines a plotting function to visualize reconstructed seafloor age and spreading-rate grids together with plate tectonic features. It overlays continents, coastlines, ridges, subduction teeth, and miscellaneous boundaries, and colors trench geometries by their associated average slab dip values. The resulting maps are saved as PNG files and provide a combined visual check of reconstructed plate boundaries, slab properties, and oceanic age or spreading-rate distributions.

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import matplotlib.cm as cm
import matplotlib.colors as colors
import numpy as np

# Function to plot seafloor grid with given dataset and colormap
def plot_seafloor_grid(grid_data, cmap, title, label, output_filename):
    fig = plt.figure(figsize=(16, 12))
    ax = fig.add_subplot(111, projection=ccrs.Mollweide(190))

    # Plot existing features
    gplot.plot_continents(ax, facecolor='0.8')
    gplot.plot_coastlines(ax, color='0.5')
    gplot.plot_ridges_and_transforms(ax, color='red')
    gplot.plot_subduction_teeth(ax, color='k')
    gplot.plot_misc_boundaries(ax, color='purple', linewidth=1.5, alpha=0.8)  # ✅ NEW


    # Define colormap and normalization for trench slab dip visualization
    trench_cmap = cm.get_cmap('viridis')  # Choose a colormap (e.g., viridis, plasma, coolwarm)
    norm = colors.Normalize(vmin=min(average_dips), vmax=max(average_dips))

    # Plot trenches with color based on slab dip values
    for i, geom in enumerate(trenches.geometry):
        if geom.geom_type == 'LineString':
            coords = np.array(geom.coords)
            color = trench_cmap(norm(average_dips[i]))  # Get color for the corresponding slab dip value
            ax.plot(coords[:, 0], coords[:, 1], transform=ccrs.PlateCarree(), color=color, linewidth=2)

        elif geom.geom_type == 'MultiLineString':
            for part in geom.geoms:
                coords = np.array(part.coords)
                color = trench_cmap(norm(average_dips[i]))  # Get color for the corresponding slab dip value
                ax.plot(coords[:, 0], coords[:, 1], transform=ccrs.PlateCarree(), color=color, linewidth=2)

    # Add a colorbar with title for slab dip values
    sm = cm.ScalarMappable(norm=norm, cmap=trench_cmap)
    sm.set_array([])  # Dummy to ensure colorbar works
    cbar = plt.colorbar(sm, ax=ax, orientation='horizontal', shrink=0.5, pad=0.05)
    cbar.set_label('Slab Dip (degrees)', fontsize=14)

    # Plot seafloor grid
    im = gplot.plot_grid(ax, grid_data, cmap=cmap, alpha=0.6)
    fig.colorbar(im, ax=ax, orientation='horizontal', shrink=0.4, pad=0.1, label=label)

    # Add title with time and model information
    ax.set_title(f"{title} - {plate_reconstruction_model} at {reconstruction_time} Ma", fontsize=16, fontweight='bold')

    # Save the updated plot with overlays
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    print(f"Updated plate reconstruction plot saved as {output_filename}")

    # Show the updated plot (optional)
    plt.show()

# Overlay the generated seafloor grids
try:
    print("Overlaying seafloor grids...")

    # Load generated seafloor age and spreading rate grids
    agegrid_filename = (
        f"{output_dir}/SEAFLOOR_AGE/"
        f"{plate_reconstruction_model}_SEAFLOOR_AGE_grid_{reconstruction_time:.2f}Ma.nc"
    )

    spreadrate_filename = (
        f"{output_dir}/SPREADING_RATE/"
        f"{plate_reconstruction_model}_SPREADING_RATE_grid_{reconstruction_time:.2f}Ma.nc"
    )

    # Load grids using GPlately's Raster object
    plate_age_grid = gplately.grids.Raster(agegrid_filename)
    plate_spread_grid = gplately.grids.Raster(spreadrate_filename)

    # Plot seafloor age map
    plot_seafloor_grid(
        plate_age_grid.data,
        cmap='YlGnBu',
        title="Seafloor Age Map",
        label="Seafloor Age (Ma)",
        output_filename=os.path.join(output_dir, f"{plate_reconstruction_model}_{reconstruction_time}Ma_age.png")
    )

    # Plot spreading rate map
    plot_seafloor_grid(
        plate_spread_grid.data,
        cmap='magma',
        title="Seafloor Spreading Rate Map",
        label="Seafloor Spreading Rate (mm/yr)",
        output_filename=os.path.join(output_dir, f"{plate_reconstruction_model}_{reconstruction_time}Ma_spreading_rate.png")
    )

    print("Seafloor grids plotted successfully.")

except Exception as e:
    print(f"Error overlaying seafloor grids: {e}")


## GeoJSON to World Builder JSON Conversion

This script converts GeoJSON files containing geological features into a World Builder-compatible JSON format for geodynamic modeling, assigning appropriate properties such as depth, thermal, and compositional characteristics. 

Plate boundaries, such as ridges and trenches, are converted into **Fault** features, which include properties like dip angle, segment length, and composition variations. Continents are represented as **Continental Plate** features, characterized by uniform composition and a defined maximum depth.

This section merges touching continental polygons and estimates a simple continent-thickness structure from polygon geometry. It computes interior points, evaluates distance from the continental margin, assigns thickness values between a minimum and maximum depth, and extracts representative isodepth points. The merged continent geometries and derived depth-control points are then saved to a new GeoJSON file for later use in World Builder continental plate parameterization.

In [ ]:
import os
import json
import numpy as np
import geopandas as gpd
from shapely.geometry import Point, mapping
import matplotlib.pyplot as plt

data_dir = "./data/"
original_continent_path = os.path.join(data_dir, "continents.geojson")
output_continent_merged_path = os.path.join(data_dir, "continents_merged.geojson")

# Load the original continents.geojson file
gdf = gpd.read_file(original_continent_path)

# Merge all touching polygons separately
merged_polygons = gdf.dissolve().explode(index_parts=True).reset_index(drop=True)

# Save merged polygons to a GeoJSON file
merged_polygons.to_file(output_continent_merged_path, driver="GeoJSON")
print("Merged continents saved to continents_merged.geojson")

# Parameters
degree_to_km = 111
transition_distance_km = 300
transition_distance_deg = transition_distance_km / degree_to_km

max_thickness = 150  # km
min_thickness = 30   # km
isodepth_value = max_thickness - 1  # 149 km (your choice)

isodepth_data = []

# -------------------------------------------------------------------
# MAIN LOOP
# -------------------------------------------------------------------
for idx, polygon in merged_polygons.iterrows():

    geom = polygon.geometry
    minx, miny, maxx, maxy = geom.bounds

    x_values = np.arange(minx, maxx, 0.25)
    y_values = np.arange(miny, maxy, 0.25)

    points = [Point(x, y) for x in x_values for y in y_values if geom.contains(Point(x, y))]

    # -------------------------------------------------------------------
    # SAFETY CHECK: polygon too small to contain grid points
    # -------------------------------------------------------------------
    if len(points) == 0:
        centroid = geom.centroid
        feature = {
            "polygon_id": idx,
            "max_depth": min_thickness * 1e3,
            "continent_geometry": mapping(geom),
            "coordinate_depth": [[centroid.x, centroid.y]],
        }
        isodepth_data.append(feature)
        continue

    # Compute distance to the polygon boundary
    distance_to_border = [geom.boundary.distance(pt) for pt in points]

    # Compute thickness values
    thickness_values = [
        min_thickness + (max_thickness - min_thickness) * min(d / transition_distance_deg, 1)
        for d in distance_to_border
    ]

    # Compute maximum depth point
    max_depth = max(thickness_values)
    max_depth_point = points[thickness_values.index(max_depth)]

    # Extract isodepth points
    isodepth_points = [
        [pt.x, pt.y] for pt, thick in zip(points, thickness_values)
        if abs(thick - isodepth_value) < 1
    ]

    # Fallback: if no isodepth match, use max-depth point
    if not isodepth_points:
        isodepth_points = [[max_depth_point.x, max_depth_point.y]]

    # Store result
    feature = {
        "polygon_id": idx,
        "max_depth": max_depth * 1e3,  # convert to meters
        "continent_geometry": mapping(geom),
        "coordinate_depth": isodepth_points
    }
    isodepth_data.append(feature)

# -------------------------------------------------------------------
# SAVE GEOJSON
# -------------------------------------------------------------------
output_path = "./data/continents_merged_with_margins.geojson"

isodepth_geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {
                "polygon_id": data["polygon_id"],
                "max_depth": data["max_depth"]
            },
            "geometry": {
                "type": "MultiPoint",
                "coordinates_depth": data["coordinate_depth"]
            },
            "continent_geometry": data["continent_geometry"]
        }
        for data in isodepth_data
    ]
}

with open(output_path, "w") as f:
    json.dump(isodepth_geojson, f, indent=4)

print(f"Isodepth points saved to {output_path}")

# -------------------------------------------------------------------
# PLOT
# -------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))
merged_polygons.plot(ax=ax, color="lightblue", edgecolor="black", alpha=0.5)

for data in isodepth_data:
    xs, ys = zip(*data["coordinate_depth"])
    ax.scatter(xs, ys, color="red", s=5)

ax.set_title("Merged Continents with Isodepth (≈150 km)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()


## Crustal thickness and topography from synthetic beta

Continental deformation is represented by a synthetic beta factor, where:

beta > 1 indicates extension and crustal thinning
beta < 1 indicates compression and crustal thickening

At each timestep, crustal thickness is updated incrementally from beta, starting from an initial reference crustal thickness. The final crustal thickness is then converted into continental topography using a simple Airy isostatic relation, assuming that thicker crust produces higher elevation and thinner crust produces lower elevation relative to a reference thickness.

An optional cumulative tectonic correction (H_correction) can also be added:

extension produces subsidence
compression produces uplift

So the workflow is:

beta → crustal thickness evolution → isostatic topography (+ optional tectonic correction)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from shapely.geometry import Point
from scipy.spatial import cKDTree
import pygplates

# ============================================================
# User parameters
# ============================================================
lookback_time = 50   # Myr before reconstruction_time
dt = 5               # Myr timestep
seed_spacing = 2.0   # degrees for continental point spacing

end_time = reconstruction_time
start_time = reconstruction_time + lookback_time

# Synthetic beta parameters
A_ext = 1.0
L_ext = 800.0   # km
A_comp = 0.4
L_comp = 400.0  # km

# Incremental update strength
beta_scale = 0.30  # ~X% of the deformation signal applied every X Myr

# Crust thickness bounds
initial_crust_km = 40.0
min_crust_km = 10.0
max_crust_km = 80.0

# ============================================================
# Optional H_correction switch
# ============================================================
apply_H_correction_to_topography = False

# Topography correction from active tectonics
tectonic_corr_rate_m_per_myr = 300.0   # meters per Myr for unit (beta-1)
tectonic_corr_step_clip_m = 1000.0     # max uplift/subsidence per timestep (m)
tectonic_corr_total_clip_m = 4000.0    # max cumulative correction (m)

# Remeshing parameters
add_threshold_deg = 1.2 * seed_spacing
inherit_threshold_deg = 1.5 * seed_spacing

DEG_TO_KM = 111.0

lagrangian_dir = os.path.join(output_dir, "synthetic_crust_lagrangian")
os.makedirs(lagrangian_dir, exist_ok=True)

time_values = list(range(start_time, end_time - 1, -dt))
if time_values[-1] != end_time:
    time_values.append(end_time)

print("Running Lagrangian crust evolution for times:", time_values)
print(f"apply_H_correction_to_topography = {apply_H_correction_to_topography}")

# Ensure we have a proper pygplates RotationModel object
rotation_model_obj = pygplates.RotationModel(rotation_model)

# ============================================================
# Helpers
# ============================================================
def extract_points_from_geoms(geoms):
    pts = []
    for geom in geoms:
        if geom is None:
            continue

        if geom.geom_type == "LineString":
            pts.extend(list(geom.coords))

        elif geom.geom_type == "MultiLineString":
            for part in geom.geoms:
                pts.extend(list(part.coords))

        elif geom.geom_type == "Polygon":
            pts.extend(list(geom.exterior.coords))

        elif geom.geom_type == "MultiPolygon":
            for part in geom.geoms:
                pts.extend(list(part.exterior.coords))

    return np.array(pts, dtype=float) if len(pts) > 0 else np.empty((0, 2), dtype=float)


def get_continent_union_at_time(gplot, time):
    gplot.time = time
    continents_gdf = gplot.get_continents()

    if continents_gdf is None or continents_gdf.empty:
        return None

    try:
        return continents_gdf.union_all()
    except AttributeError:
        return continents_gdf.unary_union


def seed_candidate_points_in_union(continent_union, spacing_deg=2.0):
    lon_vals = np.arange(-180, 180, spacing_deg)
    lat_vals = np.arange(-88, 88 + spacing_deg, spacing_deg)

    candidates = []
    for lat in lat_vals:
        for lon in lon_vals:
            pt = Point(lon, lat)
            if continent_union is not None and continent_union.covers(pt):
                candidates.append((lon, lat))

    return np.array(candidates, dtype=float) if len(candidates) > 0 else np.empty((0, 2), dtype=float)


def seed_points_inside_continents_at_time(gplot, time, spacing_deg=2.0):
    continent_union = get_continent_union_at_time(gplot, time)
    if continent_union is None:
        raise ValueError(f"No continents found at {time} Ma")

    seeded = seed_candidate_points_in_union(continent_union, spacing_deg=spacing_deg)

    if len(seeded) == 0:
        raise ValueError(f"No seed points found inside continents at {time} Ma")

    return seeded


def reconstruct_points_between_times(rotation_model, points_lonlat, from_time, to_time):
    """
    Reconstruct lon/lat points from from_time to to_time using a finite rotation.
    """
    finite_rotation = rotation_model.get_rotation(to_time, 0, from_time)

    out = []
    for lon, lat in points_lonlat:
        point = pygplates.PointOnSphere(lat, lon)
        new_point = finite_rotation * point
        new_lat, new_lon = new_point.to_lat_lon()
        out.append((new_lon, new_lat))

    return np.array(out, dtype=float)


def get_ridge_trench_points_at_time(gplot, time):
    """
    Get ridge and trench point clouds at a given reconstruction time.
    """
    gplot.time = time

    ridges_gdf = gplot.get_ridges_and_transforms()
    trenches_gdf = gplot.get_trenches()

    ridge_points = (
        extract_points_from_geoms(ridges_gdf.geometry)
        if ridges_gdf is not None and not ridges_gdf.empty
        else np.empty((0, 2))
    )
    trench_points = (
        extract_points_from_geoms(trenches_gdf.geometry)
        if trenches_gdf is not None and not trenches_gdf.empty
        else np.empty((0, 2))
    )

    return ridge_points, trench_points


def remesh_continental_points(
    point_lonlat,
    T,
    H_corr,
    continent_union,
    seed_spacing=2.0,
    add_threshold_deg=None,
    inherit_threshold_deg=None,
    initial_crust_km=40.0
):
    """
    Remove points outside current continents and add missing points inside continents.
    Carries both crust thickness T and cumulative topography correction H_corr.
    """
    if add_threshold_deg is None:
        add_threshold_deg = 1.2 * seed_spacing
    if inherit_threshold_deg is None:
        inherit_threshold_deg = 1.5 * seed_spacing

    inside_mask = np.array([
        continent_union.covers(Point(lon, lat))
        for lon, lat in point_lonlat
    ], dtype=bool)

    kept_points = point_lonlat[inside_mask]
    kept_T = T[inside_mask]
    kept_H_corr = H_corr[inside_mask]

    candidate_points = seed_candidate_points_in_union(continent_union, spacing_deg=seed_spacing)

    if len(candidate_points) == 0:
        return kept_points, kept_T, kept_H_corr

    if len(kept_points) > 0:
        kept_tree = cKDTree(kept_points)
        dist_to_existing, idx_nearest = kept_tree.query(candidate_points)
    else:
        dist_to_existing = np.full(len(candidate_points), np.inf)
        idx_nearest = np.full(len(candidate_points), -1)

    add_mask = dist_to_existing > add_threshold_deg
    new_points = candidate_points[add_mask]

    if len(new_points) == 0:
        return kept_points, kept_T, kept_H_corr

    new_T = np.full(len(new_points), initial_crust_km, dtype=float)
    new_H_corr = np.zeros(len(new_points), dtype=float)

    if len(kept_points) > 0:
        nearest_dist = dist_to_existing[add_mask]
        nearest_idx = idx_nearest[add_mask]

        inherit_mask = nearest_dist <= inherit_threshold_deg
        new_T[inherit_mask] = kept_T[nearest_idx[inherit_mask]]
        new_H_corr[inherit_mask] = kept_H_corr[nearest_idx[inherit_mask]]

    merged_points = np.vstack([kept_points, new_points])
    merged_T = np.concatenate([kept_T, new_T])
    merged_H_corr = np.concatenate([kept_H_corr, new_H_corr])

    return merged_points, merged_T, merged_H_corr


def compute_beta_on_advected_points(point_lonlat, ridge_points, trench_points):
    """
    Compute synthetic beta for advected continental points.
    """
    if len(ridge_points) > 0:
        ridge_tree = cKDTree(ridge_points)
        ridge_dist_deg, _ = ridge_tree.query(point_lonlat)
        dist_ridge_km = ridge_dist_deg * DEG_TO_KM
    else:
        dist_ridge_km = np.full(len(point_lonlat), np.nan)

    if len(trench_points) > 0:
        trench_tree = cKDTree(trench_points)
        trench_dist_deg, _ = trench_tree.query(point_lonlat)
        dist_trench_km = trench_dist_deg * DEG_TO_KM
    else:
        dist_trench_km = np.full(len(point_lonlat), np.nan)

    beta_ext = A_ext * np.exp(-dist_ridge_km / L_ext)
    beta_comp = A_comp * np.exp(-dist_trench_km / L_comp)

    beta_raw = 1.0 + beta_ext - beta_comp
    beta = np.clip(beta_raw, 0.6, 2.5)

    return beta, beta_raw, dist_ridge_km, dist_trench_km


def update_crustal_thickness(T_old, beta_slice, beta_scale):
    """
    Incremental cumulative update.
    """
    beta_increment = 1.0 + beta_scale * (beta_slice - 1.0)
    beta_increment = np.clip(beta_increment, 0.85, 1.15)

    T_new = T_old / beta_increment
    T_new = np.clip(T_new, min_crust_km, max_crust_km)

    return T_new, beta_increment


def update_tectonic_topography_correction(
    H_corr_old,
    beta_slice,
    dt,
    tectonic_corr_rate_m_per_myr=300.0,
    tectonic_corr_step_clip_m=1000.0,
    tectonic_corr_total_clip_m=4000.0
):
    """
    Incremental tectonic topography correction from synthetic beta.

    beta > 1  -> extension   -> subsidence (negative correction)
    beta < 1  -> compression -> uplift     (positive correction)
    """
    dH = -tectonic_corr_rate_m_per_myr * (beta_slice - 1.0) * dt
    dH = np.clip(dH, -tectonic_corr_step_clip_m, tectonic_corr_step_clip_m)

    H_corr_new = H_corr_old + dH
    H_corr_new = np.clip(H_corr_new, -tectonic_corr_total_clip_m, tectonic_corr_total_clip_m)

    return H_corr_new, dH


def scatter_map(lons, lats, values, title, cbar_label, out_png, gplot, time, vmin=None, vmax=None):
    fig = plt.figure(figsize=(16, 10))
    ax = fig.add_subplot(111, projection=ccrs.Mollweide(central_longitude=20))

    sc = ax.scatter(
        lons, lats,
        c=values,
        s=8,
        transform=ccrs.PlateCarree(),
        vmin=vmin,
        vmax=vmax
    )

    try:
        gplot.time = time
        gplot.plot_continents(ax, facecolor='none', edgecolor='black', linewidth=0.5)
    except Exception:
        pass
    try:
        gplot.plot_coastlines(ax, color='0.5')
    except Exception:
        pass
    try:
        gplot.plot_ridges(ax, color='red', alpha=0.4)
    except Exception:
        pass
    try:
        gplot.plot_transforms(ax, color='red', alpha=0.2)
    except Exception:
        pass
    try:
        gplot.plot_subduction_teeth(ax, color='black', alpha=0.4)
    except Exception:
        pass

    cbar = plt.colorbar(sc, ax=ax, orientation='horizontal', shrink=0.5, pad=0.08)
    cbar.set_label(cbar_label)

    ax.set_title(f"{title} - {plate_reconstruction_model} at {time} Ma")
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# Seed continental points at start_time
# ============================================================
seed_lonlat = seed_points_inside_continents_at_time(gplot, start_time, spacing_deg=seed_spacing)
print(f"Seeded {len(seed_lonlat)} continental points at {start_time} Ma")

point_lonlat = seed_lonlat.copy()
T = np.full(len(point_lonlat), initial_crust_km, dtype=float)
H_corr = np.zeros(len(point_lonlat), dtype=float)

history = []

# ============================================================
# Time loop
# ============================================================
for i, time in enumerate(time_values):
    print(f"\nProcessing time = {time} Ma")

    if i > 0:
        prev_time = time_values[i - 1]
        point_lonlat = reconstruct_points_between_times(
            rotation_model=rotation_model_obj,
            points_lonlat=point_lonlat,
            from_time=prev_time,
            to_time=time
        )

    continent_union = get_continent_union_at_time(gplot, time)
    if continent_union is None:
        print(f"No continents found at {time} Ma, skipping.")
        continue

    point_lonlat, T, H_corr = remesh_continental_points(
        point_lonlat=point_lonlat,
        T=T,
        H_corr=H_corr,
        continent_union=continent_union,
        seed_spacing=seed_spacing,
        add_threshold_deg=add_threshold_deg,
        inherit_threshold_deg=inherit_threshold_deg,
        initial_crust_km=initial_crust_km
    )

    print(f"Number of continental points after remeshing: {len(point_lonlat)}")

    ridge_points, trench_points = get_ridge_trench_points_at_time(gplot, time)

    beta_slice, beta_raw, dist_ridge_km, dist_trench_km = compute_beta_on_advected_points(
        point_lonlat=point_lonlat,
        ridge_points=ridge_points,
        trench_points=trench_points
    )

    T, beta_increment = update_crustal_thickness(
        T_old=T,
        beta_slice=beta_slice,
        beta_scale=beta_scale
    )

    # Optional H_correction update
    if apply_H_correction_to_topography:
        H_corr, dH_step = update_tectonic_topography_correction(
            H_corr_old=H_corr,
            beta_slice=beta_slice,
            dt=dt,
            tectonic_corr_rate_m_per_myr=tectonic_corr_rate_m_per_myr,
            tectonic_corr_step_clip_m=tectonic_corr_step_clip_m,
            tectonic_corr_total_clip_m=tectonic_corr_total_clip_m
        )
    else:
        dH_step = np.zeros_like(beta_slice)
        H_corr = np.zeros_like(H_corr)

    history.append({
        "time_ma": time,
        "n_points": int(len(point_lonlat)),
        "mean_crust_km": float(np.nanmean(T)),
        "min_crust_km": float(np.nanmin(T)),
        "max_crust_km": float(np.nanmax(T)),
        "mean_beta": float(np.nanmean(beta_slice)),
        "mean_H_corr_m": float(np.nanmean(H_corr)),
        "min_H_corr_m": float(np.nanmin(H_corr)),
        "max_H_corr_m": float(np.nanmax(H_corr)),
        "apply_H_correction": bool(apply_H_correction_to_topography),
    })

    print(
        f"Mean crust: {np.nanmean(T):.2f} km | "
        f"Min: {np.nanmin(T):.2f} km | "
        f"Max: {np.nanmax(T):.2f} km | "
        f"Mean Hcorr: {np.nanmean(H_corr):.2f} m"
    )

    step_df = pd.DataFrame({
        "lon": point_lonlat[:, 0],
        "lat": point_lonlat[:, 1],
        "beta_raw": beta_raw,
        "beta_slice": beta_slice,
        "beta_increment": beta_increment,
        "dist_ridge_km": dist_ridge_km,
        "dist_trench_km": dist_trench_km,
        "crustal_thickness_km": T,
        "time_ma": time,
        "dH_step_m": dH_step,
        "tectonic_topography_correction_m": H_corr,
        "apply_H_correction": apply_H_correction_to_topography,
    })

    step_csv = os.path.join(
        lagrangian_dir,
        f"{plate_reconstruction_model}_{time:03d}Ma_advected_points.csv"
    )
    step_df.to_csv(step_csv, index=False)

    scatter_map(
        point_lonlat[:, 0],
        point_lonlat[:, 1],
        beta_slice,
        "Synthetic Beta on Advected Continental Points",
        "Synthetic beta",
        os.path.join(lagrangian_dir, f"{plate_reconstruction_model}_{time:03d}Ma_beta_points.png"),
        gplot,
        time,
        vmin=0.6,
        vmax=2.0
    )

    scatter_map(
        point_lonlat[:, 0],
        point_lonlat[:, 1],
        T,
        "Cumulative Crust Thickness on Advected Continental Points",
        "Crust thickness (km)",
        os.path.join(lagrangian_dir, f"{plate_reconstruction_model}_{time:03d}Ma_crust_points.png"),
        gplot,
        time,
        vmin=min_crust_km,
        vmax=max_crust_km
    )

# ============================================================
# Save final outputs
# ============================================================
history_df = pd.DataFrame(history)
history_csv = os.path.join(lagrangian_dir, f"{plate_reconstruction_model}_advected_crust_history.csv")
history_df.to_csv(history_csv, index=False)

final_df = pd.DataFrame({
    "lon": point_lonlat[:, 0],
    "lat": point_lonlat[:, 1],
    "crustal_thickness_km": T,
    "beta_slice": beta_slice,
    "beta_raw": beta_raw,
    "tectonic_topography_correction_m": H_corr,
    "apply_H_correction": apply_H_correction_to_topography,
})
final_csv = os.path.join(
    lagrangian_dir,
    f"{plate_reconstruction_model}_{reconstruction_time}Ma_final_advected_crust_points.csv"
)
final_df.to_csv(final_csv, index=False)

scatter_map(
    point_lonlat[:, 0],
    point_lonlat[:, 1],
    beta_slice,
    "Final Synthetic Beta on Advected Continental Points",
    "Synthetic beta",
    os.path.join(lagrangian_dir, f"{plate_reconstruction_model}_{reconstruction_time}Ma_final_beta_points.png"),
    gplot,
    reconstruction_time,
    vmin=0.6,
    vmax=2.0
)

scatter_map(
    point_lonlat[:, 0],
    point_lonlat[:, 1],
    T,
    "Final Cumulative Crust Thickness on Advected Continental Points",
    "Crust thickness (km)",
    os.path.join(lagrangian_dir, f"{plate_reconstruction_model}_{reconstruction_time}Ma_final_crust_points.png"),
    gplot,
    reconstruction_time,
    vmin=min_crust_km,
    vmax=max_crust_km
)

print(f"\nSaved history to {history_csv}")
print(f"Saved final point dataset to {final_csv}")
print(history_df.head())

In [ ]:
from scipy.interpolate import griddata

# ============================================================
# Interpolate final advected point cloud to a regular grid
# ============================================================
plot_spacing = 1.0  # degrees for smoother final maps

plot_lon_vals = np.arange(-180, 180, plot_spacing)
plot_lat_vals = np.arange(-88, 88 + plot_spacing, plot_spacing)
plot_grid_lon, plot_grid_lat = np.meshgrid(plot_lon_vals, plot_lat_vals)

# Current advected point cloud at reconstruction_time
final_lons = point_lonlat[:, 0]
final_lats = point_lonlat[:, 1]
final_points_xy = np.column_stack([final_lons, final_lats])

# ------------------------------------------------------------
# Interpolate beta and crust thickness
# ------------------------------------------------------------
final_beta_grid = griddata(
    final_points_xy,
    beta_slice,
    (plot_grid_lon, plot_grid_lat),
    method="linear"
)

final_thickness_grid = griddata(
    final_points_xy,
    T,
    (plot_grid_lon, plot_grid_lat),
    method="linear"
)

# Fill holes only to stabilize interpolation
beta_grid_nearest = griddata(
    final_points_xy,
    beta_slice,
    (plot_grid_lon, plot_grid_lat),
    method="nearest"
)

thickness_grid_nearest = griddata(
    final_points_xy,
    T,
    (plot_grid_lon, plot_grid_lat),
    method="nearest"
)

final_beta_grid = np.where(np.isnan(final_beta_grid), beta_grid_nearest, final_beta_grid)
final_thickness_grid = np.where(np.isnan(final_thickness_grid), thickness_grid_nearest, final_thickness_grid)

# ------------------------------------------------------------
# Interpolate cumulative tectonic correction
# ------------------------------------------------------------
if apply_H_correction_to_topography:
    final_Hcorr_grid = griddata(
        final_points_xy,
        H_corr,
        (plot_grid_lon, plot_grid_lat),
        method="linear"
    )

    Hcorr_grid_nearest = griddata(
        final_points_xy,
        H_corr,
        (plot_grid_lon, plot_grid_lat),
        method="nearest"
    )

    final_Hcorr_grid = np.where(np.isnan(final_Hcorr_grid), Hcorr_grid_nearest, final_Hcorr_grid)
else:
    final_Hcorr_grid = np.zeros_like(plot_grid_lon, dtype=float)

# ============================================================
# Mask to continents only at final reconstruction time
# ============================================================
final_continent_union = get_continent_union_at_time(gplot, reconstruction_time)

continent_mask = np.array([
    final_continent_union.covers(Point(lon, lat))
    for lon, lat in zip(plot_grid_lon.ravel(), plot_grid_lat.ravel())
], dtype=bool).reshape(plot_grid_lon.shape)

final_beta_grid = np.where(continent_mask, final_beta_grid, np.nan)
final_thickness_grid = np.where(continent_mask, final_thickness_grid, np.nan)
final_Hcorr_grid = np.where(continent_mask, final_Hcorr_grid, np.nan)

def plot_interpolated_grid(
    grid_lon,
    grid_lat,
    values_2d,
    title,
    cbar_label,
    out_png,
    gplot,
    time,
    vmin=None,
    vmax=None,
    savefig=False
):
    fig = plt.figure(figsize=(16, 10))
    ax = fig.add_subplot(111, projection=ccrs.Mollweide(central_longitude=20))

    mesh = ax.pcolormesh(
        grid_lon,
        grid_lat,
        values_2d,
        transform=ccrs.PlateCarree(),
        shading="auto",
        vmin=vmin,
        vmax=vmax
    )

    try:
        gplot.time = time
        gplot.plot_continents(ax, facecolor='none', edgecolor='black', linewidth=0.5)
    except Exception:
        pass
    try:
        gplot.plot_coastlines(ax, color='0.5')
    except Exception:
        pass
    try:
        gplot.plot_ridges(ax, color='red', alpha=0.4)
    except Exception:
        pass
    try:
        gplot.plot_transforms(ax, color='red', alpha=0.2)
    except Exception:
        pass
    try:
        gplot.plot_subduction_teeth(ax, color='black', alpha=0.4)
    except Exception:
        pass

    cbar = plt.colorbar(mesh, ax=ax, orientation='horizontal', shrink=0.5, pad=0.08)
    cbar.set_label(cbar_label)

    ax.set_title(f"{title} - {plate_reconstruction_model} at {time} Ma")

    if savefig:
        plt.savefig(out_png, dpi=300, bbox_inches="tight")

    plt.show()


plot_interpolated_grid(
    plot_grid_lon,
    plot_grid_lat,
    final_beta_grid,
    "Final Synthetic Continental Beta",
    "Synthetic beta",
    os.path.join(
        lagrangian_dir,
        f"{plate_reconstruction_model}_{reconstruction_time}Ma_final_beta_grid.png"
    ),
    gplot,
    reconstruction_time,
    vmin=0.6,
    vmax=2.0
)

plot_interpolated_grid(
    plot_grid_lon,
    plot_grid_lat,
    final_thickness_grid,
    "Final Cumulative Continental Crust Thickness",
    "Crust thickness (km)",
    os.path.join(
        lagrangian_dir,
        f"{plate_reconstruction_model}_{reconstruction_time}Ma_final_crust_grid.png"
    ),
    gplot,
    reconstruction_time,
    vmin=min_crust_km,
    vmax=max_crust_km
)

if apply_H_correction_to_topography:
    plot_interpolated_grid(
        plot_grid_lon,
        plot_grid_lat,
        final_Hcorr_grid,
        "Final Cumulative Tectonic Topography Correction",
        "Tectonic correction (m)",
        os.path.join(
            lagrangian_dir,
            f"{plate_reconstruction_model}_{reconstruction_time}Ma_final_tectonics_correction_grid.png"
        ),
        gplot,
        reconstruction_time,
        vmin=-6000, 
        vmax=6000
    )

print("Plotted interpolated final grid maps.")

Plotted interpolated final grid maps.

In [ ]:
# ============================================================
# Load Oceanic bathymetry from seafloor age
# ============================================================
age_file = (
    f"{output_dir}/SEAFLOOR_AGE/"
    f"{plate_reconstruction_model}_SEAFLOOR_AGE_grid_unmasked_{reconstruction_time:.2f}Ma.nc"
)

if not os.path.exists(age_file):
    alt_age_file = (
        f"{output_dir}/SEAFLOOR_AGE/"
        f"{plate_reconstruction_model}_SEAFLOOR_AGE_grid_{reconstruction_time:.2f}Ma.nc"
    )
    if os.path.exists(alt_age_file):
        age_file = alt_age_file
    else:
        raise FileNotFoundError(f"Could not find age grid:\n{age_file}\n{alt_age_file}")

age_grid = gplately.grids.Raster(age_file)
print("Loaded seafloor age grid:", age_file)

age_data = age_grid.data
print("Age grid shape:", age_data.shape)

from scipy.interpolate import RegularGridInterpolator

# ============================================================
# 1. Continental topography from crust thickness
# ============================================================
C_ref = 40.0   # km, reference crust thickness at sea level

rho_m = 3300.0
rho_c = 2850.0
airy_factor = (rho_m - rho_c) / rho_m
alpha = 1.0 / airy_factor   # ~7.33 with these values

continental_topography_m = ((final_thickness_grid - C_ref) / alpha) * 1000.0
continental_topography_m = np.where(continent_mask, continental_topography_m, np.nan)

# final_Hcorr_grid is already zero everywhere if use_H_correction == False
continental_topography_m = continental_topography_m + final_Hcorr_grid

if apply_H_correction_to_topography:
    print("Applied H_correction to continental topography.")
else:
    print("H_correction not applied to continental topography.")

# Optional clipping
continental_topography_m = np.clip(continental_topography_m, -2000.0, 7000.0)

# ============================================================
# 2. Oceanic topography from seafloor age
# ============================================================
# Build source lon/lat axes from raster shape
nlat, nlon = age_data.shape
age_lon_vals = np.linspace(-180, 180, nlon)
age_lat_vals = np.linspace(-90, 90, nlat)

# Interpolator for age onto the plotting grid
age_interp = RegularGridInterpolator(
    (age_lat_vals, age_lon_vals),
    age_data,
    bounds_error=False,
    fill_value=np.nan
)

target_points_latlon = np.column_stack([plot_grid_lat.ravel(), plot_grid_lon.ravel()])
age_on_plot_grid = age_interp(target_points_latlon).reshape(plot_grid_lat.shape)

# Simple age-depth model
ridge_depth_m = -2200.0
subsidence_coeff_m = 350.0

oceanic_topography_m = ridge_depth_m - subsidence_coeff_m * np.sqrt(np.maximum(age_on_plot_grid, 0.0))

# Keep only oceans
oceanic_topography_m = np.where(~continent_mask, oceanic_topography_m, np.nan)

# ============================================================
# 3. Merge continents + oceans
# ============================================================
final_topography_m = np.where(
    continent_mask,
    continental_topography_m,
    oceanic_topography_m
)

# Optional clipping for visualization
final_topography_m = np.clip(final_topography_m, -7000.0, 6000.0)

Now let's plot the topography

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# Ocean colors
ocean = plt.cm.Blues_r(np.linspace(0.15, 1, 128))

# Land colors
land = plt.cm.terrain(np.linspace(0.25, 1, 128))

# Merge
topo_cmap = LinearSegmentedColormap.from_list(
    "custom_topo",
    np.vstack((ocean, land))
)

norm = TwoSlopeNorm(vmin=-7000, vcenter=0, vmax=6000)

def plot_topography_grid(
    grid_lon,
    grid_lat,
    topo_2d,
    title,
    out_png,
    gplot,
    time,
    use_H_correction=True,
    savefig=False
):
    fig = plt.figure(figsize=(16, 10))
    ax = fig.add_subplot(111, projection=ccrs.Mollweide(central_longitude=20))

    mesh = ax.pcolormesh(
        grid_lon,
        grid_lat,
        topo_2d,
        transform=ccrs.PlateCarree(),
        shading="auto",
        cmap=topo_cmap,
        norm=norm
    )

    try:
        gplot.time = time
        gplot.plot_continents(ax, facecolor='none', edgecolor='black', linewidth=0.5)
    except Exception:
        pass
    try:
        gplot.plot_coastlines(ax, color='0.4')
    except Exception:
        pass
    try:
        gplot.plot_ridges(ax, color='red', alpha=0.4)
    except Exception:
        pass
    try:
        gplot.plot_transforms(ax, color='red', alpha=0.2)
    except Exception:
        pass
    try:
        gplot.plot_subduction_teeth(ax, color='black', alpha=0.4)
    except Exception:
        pass

    cbar = plt.colorbar(mesh, ax=ax, orientation='horizontal', shrink=0.55, pad=0.08)
    cbar.set_label("Topography / bathymetry (m)")

    hcorr_label = "with H_correction" if use_H_correction else "without H_correction"
    ax.set_title(f"{title} - {plate_reconstruction_model} at {time} Ma ({hcorr_label})")

    if savefig:
        plt.savefig(out_png, dpi=300, bbox_inches="tight")

    plt.show()

plot_topography_grid(
    plot_grid_lon,
    plot_grid_lat,
    final_topography_m,
    "Synthetic Paleotopography",
    os.path.join(
        lagrangian_dir,
        f"{plate_reconstruction_model}_{reconstruction_time}Ma_final_paleotopography"
        f"{'_with_Hcorr' if apply_H_correction_to_topography else '_without_Hcorr'}.png"
    ),
    gplot,
    reconstruction_time,
    use_H_correction=apply_H_correction_to_topography,
    savefig=False
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator
from matplotlib.colors import LinearSegmentedColormap

# ============================================================
# 1. Interpolate seafloor age onto the plotting grid
# ============================================================
age_data = age_grid.data

# Build source lon/lat axes from raster shape
nlat, nlon = age_data.shape
age_lon_vals = np.linspace(-180, 180, nlon)
age_lat_vals = np.linspace(-90, 90, nlat)

# Interpolator for age onto the plotting grid
age_interp = RegularGridInterpolator(
    (age_lat_vals, age_lon_vals),
    age_data,
    bounds_error=False,
    fill_value=np.nan
)

target_points_latlon = np.column_stack([plot_grid_lat.ravel(), plot_grid_lon.ravel()])
age_on_plot_grid = age_interp(target_points_latlon).reshape(plot_grid_lat.shape)

# Keep age only in oceanic domain
age_on_plot_grid = np.where(~continent_mask, age_on_plot_grid, np.nan)

# ============================================================
# 2. Continental LAB proxy from topography
# ============================================================
# Simple scaling:
# lithosphere_thickness_km = L0 + alpha * topography_km
#
# You can tune these:
continental_reference_lithosphere_km = 120.0
continental_topo_scale = 8.0   # km lithosphere per km elevation

continental_topography_km = final_topography_m / 1000.0

continental_lithosphere_km = (
    continental_reference_lithosphere_km
    + continental_topo_scale * continental_topography_km
)

# Clip to reasonable continental range
continental_lithosphere_km = np.clip(continental_lithosphere_km, 80.0, 250.0)

# Apply only to continents
continental_lithosphere_km = np.where(continent_mask, continental_lithosphere_km, np.nan)

# ============================================================
# 3. Oceanic LAB proxy from seafloor age
# ============================================================
# Half-space cooling style lithosphere thickness:
# L = 2 * sqrt(kappa * t)
#
# where:
#   kappa ~ 1e-6 m^2/s
#   t in seconds
kappa = 1e-6
seconds_per_Myr = 1e6 * 365.25 * 24 * 3600

age_seconds = np.maximum(age_on_plot_grid, 0.0) * seconds_per_Myr
oceanic_lithosphere_km = 2.0 * np.sqrt(kappa * age_seconds) / 1000.0

# Clip to reasonable oceanic range
oceanic_lithosphere_km = np.clip(oceanic_lithosphere_km, 0.0, 120.0)

# Apply only to oceans
oceanic_lithosphere_km = np.where(~continent_mask, oceanic_lithosphere_km, np.nan)

# ============================================================
# 4. Merge continental + oceanic LAB
# ============================================================
final_lithosphere_km = np.where(
    continent_mask,
    continental_lithosphere_km,
    oceanic_lithosphere_km
)

# Convert to LAB depth in meters if needed for GWB
final_lab_depth_m = final_lithosphere_km * 1000.0

# ============================================================
# 5. Plot LAB / lithosphere thickness map
# ============================================================
def plot_lab_grid(grid_lon, grid_lat, lab_2d, title, out_png, gplot, time, vmin=0, vmax=250):
    fig = plt.figure(figsize=(16, 10))
    ax = fig.add_subplot(111, projection=ccrs.Mollweide(central_longitude=20))

    mesh = ax.pcolormesh(
        grid_lon,
        grid_lat,
        lab_2d,
        transform=ccrs.PlateCarree(),
        shading="auto",
        cmap="viridis",
        vmin=vmin,
        vmax=vmax
    )

    try:
        gplot.time = time
        gplot.plot_continents(ax, facecolor='none', edgecolor='black', linewidth=0.5)
    except Exception:
        pass
    try:
        gplot.plot_coastlines(ax, color='0.4')
    except Exception:
        pass
    try:
        gplot.plot_ridges(ax, color='red', alpha=0.4)
    except Exception:
        pass
    try:
        gplot.plot_transforms(ax, color='red', alpha=0.2)
    except Exception:
        pass
    try:
        gplot.plot_subduction_teeth(ax, color='black', alpha=0.4)
    except Exception:
        pass

    cbar = plt.colorbar(mesh, ax=ax, orientation='horizontal', shrink=0.55, pad=0.08)
    cbar.set_label("Lithosphere thickness / LAB depth proxy (km)")

    ax.set_title(f"{title} - {plate_reconstruction_model} at {time} Ma")
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()

plot_lab_grid(
    plot_grid_lon,
    plot_grid_lat,
    final_lithosphere_km,
    "Synthetic LAB / Lithosphere Thickness",
    os.path.join(lagrangian_dir, f"{plate_reconstruction_model}_{reconstruction_time}Ma_final_LAB.png"),
    gplot,
    reconstruction_time,
    vmin=0,
    vmax=250
)

# ============================================================
# 6. Save LAB fields
# ============================================================
lab_df = pd.DataFrame({
    "lon": plot_grid_lon.ravel(),
    "lat": plot_grid_lat.ravel(),
    "continent_mask": continent_mask.ravel(),
    "topography_m": final_topography_m.ravel(),
    "ocean_age_ma": age_on_plot_grid.ravel(),
    "continental_lithosphere_km": continental_lithosphere_km.ravel(),
    "oceanic_lithosphere_km": oceanic_lithosphere_km.ravel(),
    "lithosphere_thickness_km": final_lithosphere_km.ravel(),
    "LAB_depth_m": final_lab_depth_m.ravel()
})

lab_csv = os.path.join(
    lagrangian_dir,
    f"{plate_reconstruction_model}_{reconstruction_time}Ma_final_LAB.csv"
)
lab_df.to_csv(lab_csv, index=False)

print(f"Saved LAB proxy to {lab_csv}")

This section converts the extracted and processed GeoJSON features into a single World Builder-compatible JSON model. Depending on the selected options, it adds ambient mantle layers, oceanic lithosphere, continents with layered compositions and variable thickness, ridges or miscellaneous boundaries as faults, and trenches as subducting slabs with geometry, thermal structure, and kinematic parameters derived from the earlier reconstruction steps. The final output is a complete worldbuilder_gplates_model.json file ready to be used as input for Geodynamic World Builder.

In [ ]:
import json
import os
import geopandas as gpd
import numpy as np
from shapely.geometry import Point, Polygon, MultiPolygon
from shapely.ops import unary_union
from geopy.distance import geodesic
from joblib import Parallel, delayed

# Define input and output directories
data_dir = "./data/"
output_file = os.path.join(data_dir, "worldbuilder_gplates_model.json")

os.makedirs(os.path.dirname(output_file), exist_ok=True)

# Flags
include_trenches_as_faults = False
include_trenches_as_slabs = True
include_continents = True
use_passive_margin = True
include_ridges = False
include_ambient_mantle = True
include_llsvp_layer = True
include_misc_boundaries = False
include_lithosphere_in_oceanic_domain = True

continent_file = "continents_merged_with_margins.geojson" if use_passive_margin else "continents.geojson"

# List of GeoJSONs
geojson_files = [
    continent_file,
    "ridges_and_transforms.geojson",
    "misc_boundaries.geojson",
    "trenches_with_avg_parameters.geojson"
]

all_features = []

# --------------------------------
# Ambient mantle
# --------------------------------
if include_ambient_mantle:
    all_features.append({
        "model": "mantle layer",
        "name": "asthenosphere",
        "coordinates": [[-195, -95], [-195, 95], [195, 95], [195, -95], [-195, -95]],
        "composition models": [{"model": "uniform", "compositions": [0]}],
        "temperature models": [{"model": "uniform", "temperature": 5000}]
    })

if include_lithosphere_in_oceanic_domain:
    all_features.append({
        "model": "oceanic plate",
        "name": "oceanic lithosphere",
        "coordinates": [[-195, -95], [-195, 95], [195, 95], [195, -95], [-195, -95]],
        "temperature models": [{
            "model": "plate model constant age",
            "operation": "replace",
            "bottom temperature": 1700,
            "top temperature": 293,
            "plate age": 100e6,
            "max depth": 100e3
        }],
        "composition models": [{
            "model": "uniform",
            "operation": "replace",
            "compositions": [0],
            "max depth": 100e3
        }]
    })


# ---------------------------------------
# MAIN LOOP OVER GEOJSON FILES
# ---------------------------------------
for geojson_file in geojson_files:
    path = os.path.join(data_dir, geojson_file)
    print(f"Processing file: {path}")

    try:
        with open(path, 'r') as f:
            geojson_data = json.load(f)
    except Exception as e:
        print(f"Error reading {geojson_file}: {e}")
        continue

    if "features" not in geojson_data:
        print(f"No features in {geojson_file}")
        continue

    features = geojson_data["features"]
    print(f"Number of features found: {len(features)}")

    continental_counter = 1
    ridge_counter = 1
    trench_counter = 1
    misc_counter = 1

    # ---------------------------------------
    # LOOP THROUGH FEATURES
    # ---------------------------------------
    for feature in features:
        geometry = feature.get("geometry", {})
        properties = feature.get("properties", {})
        geometry_type = geometry.get("type", "")
        coordinates = geometry.get("coordinates", [])

        # -------------------------------------------------------------
        # CONTINENTS
        # -------------------------------------------------------------

        if geojson_file == continent_file and include_continents:

            continent_geometry = feature.get("continent_geometry", {})
            cgeom_type = continent_geometry.get("type", "")
            ccoords = continent_geometry.get("coordinates", [])
            depth_coords = geometry.get("coordinates_depth", [])

            thickness = properties.get("max_depth", 150e3)

            depth_data = (
                [[0], [thickness, depth_coords if isinstance(depth_coords[0], list)
                       else [depth_coords]]]
                if depth_coords else
                [[0], [thickness, [[0, 0]]]]
            )

            if cgeom_type == "Polygon":
                continent_coords = ccoords[0]
            elif cgeom_type == "MultiPolygon":
                continent_coords = ccoords[0][0]
            else:
                continue

            feature_data = {
                "model": "continental plate",
                "name": f"continental_plate_{continental_counter}",
                "min depth": -1e3,
                "max depth": thickness,
                "coordinates": continent_coords,
                "temperature models": [{
                    "model": "linear",
                    "operation": "replace",
                    "min depth": -1e3,
                    "max depth": depth_data
                }],
                "composition models": [
                    {
                        "model": "uniform",
                        "operation": "replace",
                        "compositions": [1],
                        "min depth": -1e3,
                        "max depth": min(25e3, thickness)
                    },
                    {
                        "model": "uniform",
                        "operation": "replace",
                        "compositions": [2],
                        "min depth": min(25e3, thickness),
                        "max depth": min(35e3, thickness)
                    },
                    {
                        "model": "uniform",
                        "operation": "replace",
                        "compositions": [3],
                        "min depth": min(35e3, thickness),
                        "max depth": thickness
                    }
                ]
            }

            all_features.append(feature_data)
            continental_counter += 1        

        # -------------------------------------------------------------
        # RIDGES
        # -------------------------------------------------------------
        elif geojson_file == "ridges_and_transforms.geojson" and include_ridges \
                and geometry_type == "LineString":

            all_features.append({
                "model": "fault",
                "name": f"ridge_{ridge_counter}",
                "coordinates": coordinates,
                "dip point": [0, 1],
                "min depth": -1000.0,
                "max depth": 100e3,
                "segments": [{"length": 100e3, "thickness": [50e3], "angle": [90, 90]}],
                "composition models": [{
                    "model": "smooth",
                    "operation": "replace",
                    "compositions": [0],
                    "side distance fault center": 25000.0
                }],
                "temperature models": [{
                    "model": "uniform",
                    "operation": "replace",
                    "temperature": 1700
                }]
            })

            ridge_counter += 1

        # -------------------------------------------------------------
        # MISC BOUNDARIES
        # -------------------------------------------------------------
        elif geojson_file == "misc_boundaries.geojson" and include_misc_boundaries:

            if geometry_type == "LineString":
                segments = [coordinates]
            elif geometry_type == "MultiLineString":
                segments = coordinates
            else:
                segments = []

            for seg in segments:
                all_features.append({
                    "model": "fault",
                    "name": f"misc_{misc_counter}",
                    "coordinates": seg,
                    "dip point": [0, 1],
                    "min depth": -1000.0,
                    "max depth": 100e3,
                    "segments": [{"length": 300e3, "thickness": [50e3], "angle": [90, 90]}],
                    "composition models": [{
                        "model": "smooth",
                        "operation": "replace",
                        "compositions": [0],
                        "side distance fault center": 50000.0
                    }],
                    "temperature models": [{
                        "model": "plate model constant age",
                        "operation": "replace",
                        "bottom temperature": 1700,
                        "top temperature": 293,
                        "plate age": 20e6,
                        "max depth": 100e3
                    }]
                })
                misc_counter += 1

        # -------------------------------------------------------------
        # TRENCHES (FAULTS OR SLABS)
        # -------------------------------------------------------------
        elif geojson_file == "trenches_with_avg_parameters.geojson":

            avg_dip = properties.get("Average_Slab_Dip", 45.0)
            avg_thickness = properties.get("Average_Slab_Thickness_km", 30.0) * 1e3
            avg_azimuth = properties.get("Average_Azimuth", 0.0)

            dip_point = properties.get("Dipping_Point", [0, 1])

            ridge_coords = properties.get("Ridge_Coordinates", [[[0, 0], [0, 0]]])

            spreading_velocity = properties.get("Spreading_Velocity_m_per_yr", [0.05])
            subducting_velocity = properties.get("Subducting_Velocity_m_per_yr", [0.05])

            spreading_velocity = spreading_velocity[0] if isinstance(spreading_velocity, list) else spreading_velocity
            subducting_velocity = subducting_velocity[0] if isinstance(subducting_velocity, list) else subducting_velocity

            # ---------------- LINESTRING ----------------
            if geometry_type == "LineString":

                if include_trenches_as_slabs:
                    all_features.append({
                        "model": "subducting plate",
                        "name": f"Slab_{trench_counter}",
                        "coordinates": coordinates,
                        "dip point": dip_point,
                        "segments": [
                            {"length": 500e3, "thickness": [avg_thickness], "angle": [avg_dip, avg_dip]},
                            {"length": 500e3, "thickness": [avg_thickness], "angle": [avg_dip, avg_dip]}
                        ],                        # "segments": [
                        #     {"length": 2000e3, "thickness": [200e3], "angle": [90, 90]},
                        #     {"length": 2000e3, "thickness": [200e3], "angle": [90, 90]}
                        # ],
                        "temperature models": [{
                            "model": "mass conserving",
                            "operation": "replace",
                            "reference model name": "half space model",
                            "density": 3300,
                            "thermal conductivity": 3.3,
                            "adiabatic heating": True,
                            "spreading velocity": abs(spreading_velocity),
                            "subducting velocity": abs(subducting_velocity),
                            "ridge coordinates": ridge_coords,
                            "coupling depth": 80e3,
                            "forearc cooling factor": 10.0,
                            "taper distance": 150e3,
                            "min distance slab top": -200e3,
                            "max distance slab top": 300e3
                        }],
                         "composition models": [
                        { "model": "uniform", "operation": "replace", "compositions": [4], "min distance slab top": -1e3, "max distance slab top": 8e3 },
                        { "model": "uniform", "operation": "replace", "compositions": [5], "min distance slab top": 8e3 }] 
                    })

                trench_counter += 1

            # ---------------- MULTILINESTRING ----------------
            elif geometry_type == "MultiLineString":

                for seg in coordinates:

                    if include_trenches_as_slabs:
                        all_features.append({
                            "model": "subducting plate",
                            "name": f"Slab_{trench_counter}",
                            "coordinates": seg,
                            "dip point": dip_point,
                            "segments": [
                                {"length": 500e3, "thickness": [avg_thickness], "angle": [avg_dip, avg_dip]},
                                {"length": 500e3, "thickness": [avg_thickness], "angle": [avg_dip, avg_dip]}
                            ],                            # "segments": [
                                # {"length": 2000e3, "thickness": [200e3], "angle": [90, 90]},
                                # {"length": 2000e3, "thickness": [200e3], "angle": [90, 90]}
                            # ],
                            "temperature models": [{
                                "model": "mass conserving",
                                "operation": "replace",
                                "reference model name": "half space model",
                                "density": 3300,
                                "thermal conductivity": 3.3,
                                "adiabatic heating": True,
                                "spreading velocity": abs(spreading_velocity),
                                "subducting velocity": abs(subducting_velocity),
                                "ridge coordinates": ridge_coords,
                                "coupling depth": 80e3,
                                "forearc cooling factor": 10.0,
                                "taper distance": 150e3,
                                "min distance slab top": -200e3,
                                "max distance slab top": 300e3
                            }],
                            "composition models": [
                                { "model": "uniform", "operation": "replace", "compositions": [4], "min distance slab top": -1e3, "max distance slab top": 8e3 },
                                { "model": "uniform", "operation": "replace", "compositions": [5], "min distance slab top": 8e3 }]       
                        })

                    trench_counter += 1


# --------------------------------
# LLSVP layer
# --------------------------------
if include_llsvp_layer:
    all_features.append({
        "model": "mantle layer",
        "name": "llsvp",
        "min depth": 2486e3,
        "coordinates": [[-195, -95], [-195, 95], [195, 95], [195, -95], [-195, -95]],
        "composition models": [{
            "model": "uniform",
            "operation": "replace",
            "compositions": [0]
        }],
        "temperature models": [{
            "model": "uniform",
            "operation": "replace",
            "temperature": 5000
        }]
    })


# --------------------------------
# Save JSON
# --------------------------------
with open(output_file, 'w') as out_f:
    json.dump({
        "version": "1.1",
        "interpolation": "continuous monotone spline",
        "surface temperature": 293,
        "potential mantle temperature": 1700,
        "coordinate system": {
            "model": "spherical",
            "depth method": "starting point"
        },
        "features": all_features
    }, out_f, indent=4)

print(f"\nConversion successful! Output saved to {output_file}")


This section post-processes the World Builder JSON to correct slab thermal parameters at trenches. It matches subducting plates with trench data, estimates trench age from ridge distance and subducting velocity, and replaces the spreading velocity accordingly. This avoids unrealistically young or negative trench ages when subduction is too fast relative to ridge distance.

In [ ]:
import json
import os

def load_json(filepath):
    """Load a JSON file."""
    with open(filepath, 'r') as f:
        return json.load(f)

def save_json(filepath, data):
    """Save a JSON file."""
    with open(filepath, 'w') as f:
        json.dump(data, f, indent=4)

def calculate_trench_age(distance_ridge, subducting_velocity):
    """Calculate trench age using subducting velocity instead of spreading velocity."""
    if subducting_velocity <= 0:
        return None  # Avoid division by zero
    
    age_at_trench = distance_ridge / subducting_velocity  # Age in years
    return age_at_trench

# Load the JSON files
worldbuilder_path = "./data/worldbuilder_gplates_model.json"
trenches_path = "./data/trenches_with_avg_parameters.geojson"
output_path = "./data/worldbuilder_gplates_model_with_age_correction.json"

worldbuilder_data = load_json(worldbuilder_path)
trenches_data = load_json(trenches_path)

corrected_data = []

# Iterate over World Builder features
for feature in worldbuilder_data["features"]:
    if feature["model"] == "subducting plate":
        temp_model = feature.get("temperature models", [{}])[0]
        spreading_velocity = temp_model.get("spreading velocity", 0)
        subducting_velocity = temp_model.get("subducting velocity", 0)
        
        # Match corresponding trench feature
        matched_trench = None
        for trench in trenches_data["features"]:
            trench_props = trench["properties"]
            if (trench_props["Spreading_Velocity_m_per_yr"][0] == spreading_velocity and
                trench_props["Subducting_Velocity_m_per_yr"][0] == subducting_velocity):
                matched_trench = trench
                break
        
        if matched_trench:
            distance_ridge = matched_trench["properties"].get("Minimum_Ridge_to_Trench_Distance_km", [0])[0] * 1e3  # Convert km to meters
            
            # Compute age using subducting velocity instead
            age_at_trench = calculate_trench_age(distance_ridge, subducting_velocity)
            
            # Print results
            print(f"Feature: {feature['name']}")
            print(f"  Distance Ridge to Trench: {distance_ridge / 1e3:.2f} km")
            print(f"  Spreading Velocity (Ignored): {spreading_velocity:.4f} m/yr")
            print(f"  Subducting Velocity (Used): {subducting_velocity:.4f} m/yr")
            print(f"  Age at Trench: {age_at_trench:.2f} years")
            
            # Apply correction by replacing the spreading velocity
            temp_model["spreading velocity"] = subducting_velocity  # Replace spreading rate
            
            print(f"  🔄 Corrected Spreading Velocity: {temp_model['spreading velocity']:.4f} m/yr")
            print("-----------------------------")
    
    corrected_data.append(feature)

# Save corrected JSON
save_json(output_path, {
    "version": "1.1",
    "surface temperature":293,
    "potential mantle temperature":1700,
    "interpolation": "continuous monotone spline",
    "coordinate system": {
        "model": "spherical",
        "depth method": "starting point"
    },
    "features": corrected_data
})

print(f"✅ Corrected model saved to {output_path}")


Make a plot

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import matplotlib.cm as cm
import matplotlib.colors as colors
import numpy as np

# Function to plot seafloor grid with given dataset and colormap
def plot_seafloor_grid(grid_data, cmap, title, label, output_filename):
    fig = plt.figure(figsize=(16, 12))
    ax = fig.add_subplot(111, projection=ccrs.Mollweide(190))

    # Plot existing features
    gplot.plot_continents(ax, facecolor='0.8')
    gplot.plot_coastlines(ax, color='0.5')
    gplot.plot_ridges_and_transforms(ax, color='red')
    gplot.plot_subduction_teeth(ax, color='k')

    # Define colormap and normalization for trench slab dip visualization
    trench_cmap = cm.get_cmap('viridis')  # Choose a colormap (e.g., viridis, plasma, coolwarm)
    norm = colors.Normalize(vmin=min(average_dips), vmax=max(average_dips))

    # Plot trenches with color based on slab dip values
    for i, geom in enumerate(trenches.geometry):
        if geom.geom_type == 'LineString':
            coords = np.array(geom.coords)
            color = trench_cmap(norm(average_dips[i]))  # Get color for the corresponding slab dip value
            ax.plot(coords[:, 0], coords[:, 1], transform=ccrs.PlateCarree(), color=color, linewidth=2)

        elif geom.geom_type == 'MultiLineString':
            for part in geom.geoms:
                coords = np.array(part.coords)
                color = trench_cmap(norm(average_dips[i]))  # Get color for the corresponding slab dip value
                ax.plot(coords[:, 0], coords[:, 1], transform=ccrs.PlateCarree(), color=color, linewidth=2)

    # Add a colorbar with title for slab dip values
    sm = cm.ScalarMappable(norm=norm, cmap=trench_cmap)
    sm.set_array([])  # Dummy to ensure colorbar works
    cbar = plt.colorbar(sm, ax=ax, orientation='horizontal', shrink=0.5, pad=0.05)
    cbar.set_label('Slab Dip (degrees)', fontsize=14)

    # Plot seafloor grid
    im = gplot.plot_grid(ax, grid_data, cmap=cmap, alpha=0.6)
    fig.colorbar(im, ax=ax, orientation='horizontal', shrink=0.4, pad=0.1, label=label)

    # Add title with time and model information
    ax.set_title(f"{title} - {plate_reconstruction_model} at {reconstruction_time} Ma", fontsize=16, fontweight='bold')

    # Save the updated plot with overlays
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    print(f"Updated plate reconstruction plot saved as {output_filename}")

    # Show the updated plot (optional)
    plt.show()

# Overlay the generated seafloor grids
def find_grid_file(base):
    candidates = [
        f"{base}_{reconstruction_time:.2f}Ma.nc",
        f"{base}_{reconstruction_time}Ma.nc",
        f"{base}_{reconstruction_time:.1f}Ma.nc",
        f"{base}_{reconstruction_time:.0f}Ma.nc",
        f"{base}_{reconstruction_time}.nc"
    ]
    for fn in candidates:
        full = os.path.join(output_dir, fn)
        if os.path.exists(full):
            print(f"✔ Found grid file: {full}")
            return full
    raise FileNotFoundError(
        "❌ Expected grid file not found. Tried:\n" +
        "\n".join(os.path.join(output_dir, fn) for fn in candidates)
    )

try:
    print("Overlaying seafloor grids...")

    agegrid_filename = find_grid_file(f"{plate_reconstruction_model}_SEAFLOOR_AGE_grid")
    spreadrate_filename = find_grid_file(f"{plate_reconstruction_model}_SPREADING_RATE_grid")

    plate_age_grid = gplately.grids.Raster(agegrid_filename)
    plate_spread_grid = gplately.grids.Raster(spreadrate_filename)

    plot_seafloor_grid(
        plate_age_grid.data,
        cmap='YlGnBu',
        title="Seafloor Age Map",
        label="Seafloor Age (Ma)",
        output_filename=os.path.join(output_dir, f"{plate_reconstruction_model}_{reconstruction_time}Ma_age.png")
    )

    plot_seafloor_grid(
        plate_spread_grid.data,
        cmap='magma',
        title="Seafloor Spreading Rate Map",
        label="Seafloor Spreading Rate (mm/yr)",
        output_filename=os.path.join(output_dir, f"{plate_reconstruction_model}_{reconstruction_time}Ma_spreading_rate.png")
    )

    print("Seafloor grids plotted successfully.")

except Exception as e:
    print(f"Error overlaying seafloor grids: {e}")


### Example grid file for running World Builder

The following example shows the content of a simple spherical grid file that can be used to run World Builder. It defines the grid type, dimensionality, number of compositions, spatial extent, and grid resolution in longitude, latitude, and radius.

```text
# output variables
grid_type = sphere
dim = 3
compositions = 1

# domain of the grid
x_max = 180
x_min = -180
y_max = 90
y_min = -90
z_min = 6000000
z_max = 6371000

# grid properties
n_cell_x = 60  # Set this to match n_cell_y
n_cell_y = 60  # Ensures the grid is square for sphere
n_cell_z = 

Run
gwb-grid worldbuilder_gplates_model_with_age_correction.json gwb_gplate.grid

to visualize the global model.

# Running a regional model
This section trims a global World Builder model to a selected regional latitude–longitude window for use in chunk or other regional simulations. It keeps only features that intersect the specified bounds, clips their coordinates to the regional box, and also updates associated depth-control entries and ridge coordinates where needed. The resulting JSON file provides a regionally restricted World Builder model that can be used as input for regional geodynamic setups.

In [14]:
import json
from shapely.geometry import Polygon, LineString

def load_json(filepath):
    with open(filepath, "r") as f:
        return json.load(f)

def save_json(filepath, data):
    with open(filepath, "w") as f:
        json.dump(data, f, indent=4)

def extract_all_coords(coords):
    if isinstance(coords[0], (float, int)):
        return [coords]
    return [pt for sub in coords for pt in extract_all_coords(sub)]

def is_within_bounds(coords, lat_bounds, lon_bounds):
    flat_coords = extract_all_coords(coords)
    for lon, lat in flat_coords:
        if lon_bounds[0] <= lon <= lon_bounds[1] and lat_bounds[0] <= lat <= lat_bounds[1]:
            return True
    return False

# def filter_max_depth_entries(max_depth, lat_bounds, lon_bounds):
#     filtered = []
#     for entry in max_depth:
#         if isinstance(entry, list) and len(entry) == 2 and isinstance(entry[1], list):
#             lon, lat = entry[1][0]
#             if lon_bounds[0] <= lon <= lon_bounds[1] and lat_bounds[0] <= lat <= lat_bounds[1]:
#                 filtered.append(entry)
#         else:
#             filtered.append(entry)
#     return filtered if filtered else [[150e3]]

def filter_max_depth_entries(max_depth, lat_bounds, lon_bounds):
    if isinstance(max_depth, (int, float)):
        return max_depth  # leave as-is if it's a single value

    filtered = []
    for entry in max_depth:
        if isinstance(entry, list) and len(entry) == 2 and isinstance(entry[1], list):
            lon, lat = entry[1][0]
            if lon_bounds[0] <= lon <= lon_bounds[1] and lat_bounds[0] <= lat <= lat_bounds[1]:
                filtered.append(entry)
        else:
            filtered.append(entry)

    return filtered if filtered else [[150e3]]

def trim_coordinates_to_bounds(coords, lat_bounds, lon_bounds):
    def clamp_point(pt):
        lon = max(lon_bounds[0], min(pt[0], lon_bounds[1]))
        lat = max(lat_bounds[0], min(pt[1], lat_bounds[1]))
        return [lon, lat]

    if isinstance(coords[0], (float, int)):
        return clamp_point(coords)
    elif isinstance(coords[0][0], (float, int)):
        return [clamp_point(pt) for pt in coords]
    else:
        return [[clamp_point(pt) for pt in sublist] for sublist in coords]

def clip_feature_to_bounds_with_trim(feature, lat_bounds, lon_bounds):
    coords = feature.get("coordinates", [])
    if not coords:
        return None

    trimmed_coords = trim_coordinates_to_bounds(coords, lat_bounds, lon_bounds)
    feature["coordinates"] = trimmed_coords

    # Also handle ridge coordinates if present
    for model in feature.get("temperature models", []):
        if model.get("model") == "mass conserving" and "ridge coordinates" in model:
            model["ridge coordinates"] = [
                trim_coordinates_to_bounds(ridge, lat_bounds, lon_bounds)
                for ridge in model["ridge coordinates"]
            ]

    for model_type in ["temperature models", "composition models"]:
        for model in feature.get(model_type, []):
            if "max depth" in model:
                model["max depth"] = filter_max_depth_entries(
                    model["max depth"], lat_bounds, lon_bounds
                )
    return feature

def filter_and_trim_features(data, lat_bounds, lon_bounds):
    new_features = []
    for feature in data["features"]:
        coords = feature.get("coordinates", [])
        if not coords:
            continue
        if is_within_bounds(coords, lat_bounds, lon_bounds):
            trimmed = clip_feature_to_bounds_with_trim(feature, lat_bounds, lon_bounds)
            if trimmed:
                new_features.append(trimmed)
    return {
        "version": data["version"],
        "interpolation": data["interpolation"],
        "coordinate system": data["coordinate system"],
        "features": new_features
    }

# File paths and bounds
input_json_path = "./data/worldbuilder_gplates_model_with_age_correction.json"
output_json_path = "./data/worldbuilder_clipped_latlon.json"
lat_bounds = (34, 52)
lon_bounds = (4, 32)

# Load, trim, and save
data = load_json(input_json_path)
trimmed_data = filter_and_trim_features(data, lat_bounds, lon_bounds)
save_json(output_json_path, trimmed_data)


For running a box model, the regional chunk geometry needs to be flattened. This can be done by converting the spherical coordinates to Cartesian coordinates.

In [ ]:
import math
import json

def load_json(filepath):
    with open(filepath, "r") as f:
        return json.load(f)

def save_json(filepath, data):
    with open(filepath, "w") as f:
        json.dump(data, f, indent=4)

def extract_all_coords(coords):
    """Recursively extract all [lon, lat] points from nested coordinates."""
    if isinstance(coords[0], (float, int)):
        return [coords]
    points = []
    for item in coords:
        points.extend(extract_all_coords(item))
    return points

def latlon_to_cartesian_coords(coords, origin_lon, origin_lat):
    """Convert lat/lon to Cartesian x/y relative to origin."""
    R = 6371000  # Earth radius in meters

    def project(lon, lat):
        lon_rad = math.radians(lon)
        lat_rad = math.radians(lat)
        origin_lon_rad = math.radians(origin_lon)
        origin_lat_rad = math.radians(origin_lat)
        x = R * (lon_rad - origin_lon_rad) * math.cos(origin_lat_rad)
        y = R * (lat_rad - origin_lat_rad)
        return [x, y]

    if isinstance(coords[0], (float, int)):
        return project(coords[0], coords[1])
    elif isinstance(coords[0][0], (float, int)):
        return [project(lon, lat) for lon, lat in coords]
    else:
        return [latlon_to_cartesian_coords(sub, origin_lon, origin_lat) for sub in coords]

def convert_model_to_cartesian(data):
    """Convert all feature coordinates from lat/lon to local Cartesian."""
    all_coords = []

    for feature in data["features"]:
        all_coords.extend(extract_all_coords(feature.get("coordinates", [])))

        if "dip point" in feature:
            all_coords.append(feature["dip point"])

        for model in feature.get("temperature models", []):
            if model.get("model") == "mass conserving" and "ridge coordinates" in model:
                for ridge in model["ridge coordinates"]:
                    all_coords.extend(extract_all_coords(ridge))

        for model_type in ["temperature models", "composition models"]:
            for model in feature.get(model_type, []):
                if "max depth" in model:
                    max_depth = model["max depth"]
                    if isinstance(max_depth, list):
                        for entry in max_depth:
                            if (
                                isinstance(entry, list) and
                                len(entry) == 2 and
                                isinstance(entry[1], list) and
                                len(entry[1]) > 0
                            ):
                                all_coords.append(entry[1][0])

    if not all_coords:
        print("⚠️ No coordinates found.")
        return data

    lons = [pt[0] for pt in all_coords]
    lats = [pt[1] for pt in all_coords]
    min_lon, min_lat = min(lons), min(lats)

    cartesian_all_coords = []

    for feature in data["features"]:
        coords = feature.get("coordinates", [])
        if coords:
            cart_coords = latlon_to_cartesian_coords(coords, min_lon, min_lat)
            feature["coordinates"] = cart_coords
            cartesian_all_coords.extend(extract_all_coords(cart_coords))

        if "dip point" in feature:
            lon, lat = feature["dip point"]
            feature["dip point"] = latlon_to_cartesian_coords((lon, lat), min_lon, min_lat)

        for model in feature.get("temperature models", []):
            if model.get("model") == "mass conserving" and "ridge coordinates" in model:
                model["ridge coordinates"] = [
                    latlon_to_cartesian_coords(ridge, min_lon, min_lat)
                    for ridge in model["ridge coordinates"]
                ]

        for model_type in ["temperature models", "composition models"]:
            for model in feature.get(model_type, []):
                if "max depth" in model:
                    max_depth = model["max depth"]
                    if isinstance(max_depth, list):
                        for entry in max_depth:
                            if (
                                isinstance(entry, list) and
                                len(entry) == 2 and
                                isinstance(entry[1], list) and
                                len(entry[1]) > 0
                            ):
                                lon, lat = entry[1][0]
                                entry[1][0] = latlon_to_cartesian_coords((lon, lat), min_lon, min_lat)

    xs = [pt[0] for pt in cartesian_all_coords]
    ys = [pt[1] for pt in cartesian_all_coords]
    max_x_km = max(xs) / 1000
    max_y_km = max(ys) / 1000

    print(f"✅ Converted with origin at ({min_lon}, {min_lat})")
    print(f"✅ Max X: {max_x_km:.2f} km")
    print(f"✅ Max Y: {max_y_km:.2f} km")

    data["coordinate system"] = {"model": "cartesian"}


    return data

# Load, convert, and save
input_json_path = "./data/worldbuilder_clipped_latlon.json"
output_json_path = "./data/worldbuilder_cartesian.json"

data = load_json(input_json_path)
cartesian_data = convert_model_to_cartesian(data)
save_json(output_json_path, cartesian_data)


Run:
gwb-grid worldbuilder_gplates_model_with_age_correction.json gwb_gplate.grid

Use the desired spatial extent of your model above to define the grid file (bounds and resolution), ensuring it matches the region or global domain you want to simulate.